# Fleet Observatory: collect, investigate, then scale

**One notebook. Start offline, then 7 → 14 → 30 days.** All collection code is readable in the appendices and embedded here; no companion download is needed.

Think of two books: **attendance** tells us who was selected; **payment receipts** tell us what distributions were recorded. The existing website has a completed 30-day attendance/session sample. Its payout evidence covers **1,652 of 4,080 sampled sessions (40.5%)**. Nineteen of the 34 observed Application/service groups had no indexed monthly settlements at the fixed cutoff. Those absent outcomes remain unknown.

This notebook expands the **payment-receipt book** and investigates one covered and one uncovered Ethereum Application group. It does not silently turn unknown sessions into zero or overwrite the website. Collected earnings are explicitly conditional on **indexed settled sessions**, not all requests, all network activity or guaranteed actual income.

The source is the published fleet analyser commit `fc0eb630771b96e157caa38cd687f867dc152dd2`. Its original HTTP/endpoint collector and proved settlement collector are included **unchanged**. The staged runner and on-chain investigation are new.


## 1. Start here

1. Make a dedicated folder such as `C:\Users\mwil8\Documents\pokt-staged-collection`. Save this notebook there. Keep one copy in that folder.
2. Open it in your existing Jupyter installation. Python **3.10 or newer** works; your existing Python 3.12 is suitable. This notebook needs no extra packages, wallet, API key, Node.js, GitHub or Vercel setup.
3. Leave `MODE = "offline"` below. Select **Run → Run All Cells**. Expected: the source integrity checks and **62 tests pass**, the included reviewed Ethereum pilot appears, and no live requests are made. Offline execution should normally take seconds; folder location does not speed up remote API calls.
4. Save the notebook. Change only MODE to `"7d"`, then run all cells. By default this reuses the successful Ethereum proof and runs the new paired live investigation, rather than downloading the same pilot again.
5. Inspect the investigation output. If its technical status is `passed`, move to `"14d"`, then finally `"30d"` after the 14-day receipt says `reconciled`. Each later stage checks its prerequisites before starting HTTP. If a source discrepancy or access error appears, keep MODE at the failing stage and send the generated `review-results.zip` here.

**Use one collector at a time.** Keep the computer awake for live stages. The eight-service 14- and 30-day runs can take hours: these download all indexed settlements and check every historical settled-session roster, rather than taking four daily samples. A heartbeat prints real checkpoint progress every 30 seconds.

The work folder is `pokt-staged-work` beside the notebook. Keep it between runs. Interrupt a live cell only once, wait for it to stop, and rerun the same mode to resume. A forced kernel crash can leave a lock; its error explains which file to remove after confirming no collector is still running.

If you already have the earlier settlement cache or extracted checkpoint, set CACHE_ROOT_OVERRIDE to its exact `cache` folder. **Do not select the original sampled-collection cache.** The 30-day stage uses the original monthly plan/run ID, retaining compatible saved claims; shared historical HTTP requests also remain reusable. The 14-day claim query has a different window, so its payout pages may need fetching separately.


In [ ]:
from pathlib import Path
import sys

MODE = "offline"          # "offline", "investigate", "7d", "14d", "30d"
NOTEBOOK_PATH = None       # set the full notebook path only if automatic discovery fails
CACHE_ROOT_OVERRIDE = None # optionally Path(r"C:\...\pokt-settlement-work\cache")
REUSE_INCLUDED_PILOT = True
MAKE_REVIEW_ZIP = True     # creates a local review file; nothing is sent or published

if sys.version_info < (3, 10):
    raise RuntimeError("Python 3.10+ is required; Python 3.12 is suitable.")
if MODE not in {"offline", "investigate", "7d", "14d", "30d"}:
    raise ValueError("Choose offline, investigate, 7d, 14d or 30d")
print("Python:", sys.version.split()[0], "· MODE:", MODE)
print("Jupyter working folder:", Path.cwd())


## 2. Verify and unpack the exact code

The complete Python source is shown at the end. The suffix list, frozen overlap reference, parser fixtures and verified pilot are stored in notebook metadata to keep ordinary cells short. Setup checks SHA-256 values and preserves exact bytes on Windows. A locally edited source file is kept and triggers a clear integrity error; use a fresh work folder rather than silently overwriting it.


In [ ]:
import base64
import hashlib
import json

PACKAGE_ID = "pokt-staged-settlement-20261013"
if NOTEBOOK_PATH is None:
    candidates = []
    for candidate in Path.cwd().glob("pokt*collection*.ipynb"):
        try:
            saved = json.loads(candidate.read_text(encoding="utf-8"))
            if saved.get("metadata", {}).get("pokt_staged_collection", {}).get("packageId") == PACKAGE_ID:
                candidates.append(candidate)
        except (ValueError, OSError):
            pass
    if len(candidates) != 1:
        raise FileNotFoundError("Set NOTEBOOK_PATH to this saved notebook's full path, then rerun all cells.")
    NOTEBOOK_PATH = candidates[0]
NOTEBOOK_PATH = Path(NOTEBOOK_PATH).expanduser().resolve()
notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
metadata = notebook["metadata"]["pokt_staged_collection"]
if metadata["packageId"] != PACKAGE_ID:
    raise ValueError("Wrong notebook edition")
WORK_DIR = NOTEBOOK_PATH.parent/"pokt-staged-work"
WORK_DIR.mkdir(parents=True, exist_ok=True)

def materialise(relative, data, expected):
    if hashlib.sha256(data).hexdigest() != expected:
        raise ValueError("Embedded checksum mismatch: " + relative)
    target = (WORK_DIR/relative).resolve()
    if WORK_DIR.resolve() not in target.parents:
        raise ValueError("Invalid embedded path")
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() != expected:
        raise FileExistsError("Local source differs: " + str(target) + ". Keep your edits and use a fresh notebook folder.")
    if not target.exists():
        target.write_bytes(data)

for item in notebook["cells"]:
    spec = item.get("metadata", {}).get("embedded_module")
    if spec:
        code = "".join(item["source"]).split("\n```python\n", 1)[1].removesuffix("\n```")
        materialise(spec["path"], code.encode("utf-8"), spec["sha256"])
for relative, asset in metadata["assets"].items():
    materialise(relative, base64.b64decode(asset["base64"], validate=True), asset["sha256"])
CACHE_ROOT = Path(CACHE_ROOT_OVERRIDE).expanduser().resolve() if CACHE_ROOT_OVERRIDE is not None else WORK_DIR/"cache"
print("Work folder:", WORK_DIR)
print("Cache folder:", CACHE_ROOT)
print("Published source:", metadata["sourceCommit"])
print("Included verified pilot: Ethereum · 7 days · 26,888 claims · 1,327 sessions")


## 3. Run the offline checks

The tests cover exact payments, unknown outcomes, pagination, timeout handling, resume, historical memberships, lossless packing, stage prerequisites and real ABCI event parsing. They also test that an on-chain/indexer discrepancy blocks expansion and that a negative small probe never becomes a whole-group zero.

These checks validate the code. They do not establish fresh live API access or complete network coverage.


In [ ]:
import subprocess

checked = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py"],
    cwd=WORK_DIR, capture_output=True, text=True, timeout=120,
)
print(checked.stdout)
print(checked.stderr)
if checked.returncode:
    raise RuntimeError("Offline checks failed. Save this output before running a live mode.")


## 4. Understand the stages

| MODE | Exact UTC session-end window | Services | What happens |
|---|---|---|---|
| `offline` | Included proof only | Ethereum | Tests and reviews existing evidence; no HTTP |
| `investigate` | 29 September–5 October 2026 | Ethereum | Runs only the paired source investigation |
| `7d` | 29 September–5 October 2026 | Ethereum | Reuses or reproduces the seven-day proof, then investigates |
| `14d` | 22 September–5 October 2026 | All eight | Collects and reconciles a fortnight after the seven-day gates |
| `30d` | 6 September–5 October 2026 | All eight | Collects the full month after the fortnight gate |

All stages preserve the original fixed settlement cutoff **955,836**, eight-service choice and protocol interpretation. The month remains September 6 through October 5, not a new rolling month. We are fetching historical records, so no waiting for future days is necessary.

Each service includes all indexed Applications. Payout pagination counts before/after, records pages atomically and checks duplicates. Supplier income uses **relay/global Supplier-shareholder distribution reasons**, not the whole claim value, with exact integer uPOKT. Every settled session has its authoritative historical selected roster, owner and service-specific endpoints verified.

The investigation compares three covered/uncovered session pairs at the same historical end heights. It checks the covered claims' exact Supplier distributions against real on-chain events and looks for uncovered-group events in those same settlement blocks. It also compares the group's seven-day counts at the original and a separately recorded later cutoff.

**Limits:** the live probe is capped at twelve RPC blocks. A positive mismatch can establish an indexer discrepancy. Finding no payments in these checked blocks cannot certify every session in a group or prove no traffic. A technically successful investigation permits collection of the explicitly limited indexed-settlement universe; the original broader unknown outcomes remain unknown.

Reference sources: [Pocket sessions, claims and proofs](https://docs.pocket.network/protocol/sessions-claims-proofs/) and [tokenomics](https://docs.pocket.network/protocol/tokenomics-deep-dive/). Actual event parsing is replay-tested against an existing recorded block. Current documentation's live parameter values are not substituted for historical settings.


In [ ]:
sys.path.insert(0, str(WORK_DIR/"collector"))
sys.path.insert(0, str(WORK_DIR/"analysis"))
# A different notebook opened earlier in this kernel must not supply stale modules.
for name in ("staged_collection", "settlement_collect", "session_catalog", "pokt_live_collector", "settlement_analysis", "settlement_product"):
    loaded = sys.modules.get(name)
    if loaded and WORK_DIR.resolve() not in Path(loaded.__file__).resolve().parents:
        del sys.modules[name]
import staged_collection as staged
from pokt_live_collector import atomic_json, read_json
from settlement_analysis import analyze
from settlement_product import export as export_product

REFERENCE_PATH = WORK_DIR/"collector/settlement-reference.json.gz"
INCLUDED_PILOT = WORK_DIR/"reviewed-pilot/result.json"
PILOT_REPORT = INCLUDED_PILOT if REUSE_INCLUDED_PILOT else staged.run_dir(CACHE_ROOT, "7d")/"result.json"
included = staged.verify_receipt(INCLUDED_PILOT, "7d")
print("Included proof checksum and scope: PASS")
if MODE in {"7d", "14d", "30d"}:
    print(json.dumps(staged.plan(MODE), indent=2))
    print("Stage work:", staged.run_dir(CACHE_ROOT, MODE))
print("Original sampled payout coverage stays:", "1,652 / 4,080 · 40.5%")


## 5. Run the selected mode, or resume it

This is the live cell. Leave it running until it finishes or returns an error. In offline mode it makes no requests. The seven-day default reuses the included proof; set REUSE_INCLUDED_PILOT to False only if you want to reproduce that proof yourself.

Long live stages print downloaded claims and verified historical rosters every 30 seconds. A flat count can mean a slow request is pending. Changing folder does not fix a busy API. Requests have bounded retries; HTTP 429 means rate limiting, HTTP 403 may be an access restriction, and a transport timeout means the request did not finish in time. HTTP 200 can still contain GraphQL errors.

The previous monthly run encountered PostgreSQL statement timeout **57014**. The included collector avoids repeating expensive whole-window counts in each payout page and retries that specific timeout at the same cursor with smaller pages. Other schema/semantic errors stop. Counts still reconcile separately. Keep the exact error, rather than filling unknown amounts or changing dates to pass.

Errors are saved in a concise local file and the notebook continues to the review export. A failed stage does not create a successful new receipt. The next stage checks actual artifact bytes and prerequisites before starting collection.


In [ ]:
RESULT = None
LAST_ERROR = None

def progress(message):
    print(message, flush=True)

if MODE == "offline":
    print("OFFLINE: no live requests. Reviewing the included proof only.")
else:
    try:
        RESULT = staged.execute(MODE, CACHE_ROOT, REFERENCE_PATH, PILOT_REPORT,
                                reuse_pilot=REUSE_INCLUDED_PILOT, progress=progress)
        print("Selected mode returned:", MODE)
    except Exception as exc:
        LAST_ERROR = {"mode": MODE, "type": type(exc).__name__, "message": str(exc)}
        atomic_json(WORK_DIR/"last-error.json", LAST_ERROR)
        print("STOPPED:", LAST_ERROR["type"], LAST_ERROR["message"])
        print("Keep the work folder. Save this notebook and share review-results.zip before changing stage.")


## 6. Read the investigation separately from collection

`passed` means the paired technical checks passed for their stated scope. It does not mean every Application's missing payments were explained. An indexer discrepancy, a changed original-cutoff reference, an unavailable RPC archive or a control mismatch produces `needs_review` and blocks the later stages.

If no uncovered events were found, we retain that limited observation and the unresolved whole-group question. A larger group-wide chain reconciliation or a different reliable source may be needed. This notebook makes that distinction visible instead of declaring 100% network coverage.


In [ ]:
DIAGNOSIS = None
diagnosis_path = CACHE_ROOT/"investigation/result.json"
if RESULT and RESULT.get("investigation") is not None:
    DIAGNOSIS = RESULT["investigation"]
elif diagnosis_path.is_file():
    DIAGNOSIS = read_json(diagnosis_path)
    print("Showing the saved investigation:", DIAGNOSIS.get("completedAt"))
if DIAGNOSIS is None:
    print("No live missing-payment investigation yet. MODE=7d runs it next.")
else:
    print("Technical status:", DIAGNOSIS["technicalStatus"])
    print(DIAGNOSIS["missingPaymentFinding"])
    print("Whole-group cause established:", DIAGNOSIS["wholeGroupCauseEstablished"])
    print("Later cutoff (separate from the product):", DIAGNOSIS.get("laterCutoff"))
    for group in DIAGNOSIS["groups"]:
        print(group["kind"], "·", group["applicationId"],
              "· original-cutoff claims:", group["originalCutoffClaims"],
              "· later-cutoff claims:", group["laterCutoffClaims"])
    for error in DIAGNOSIS["errors"]:
        print("SOURCE ERROR:", error["type"], error["message"])
    print(DIAGNOSIS["scopeNotice"])


## 7. Inspect verified recorded earnings

The **offline view is the included reviewed pilot**, not a newly completed local collection. Live evidence appears here only when the selected collection returns a reconciled receipt with matching artifact bytes.

Choose owner, domain or Supplier below. Owner groups follow recorded historical ownership. Domain groups follow service-specific endpoint associations and may overlap; they do not prove a common business. Counts refer to Suppliers selected in these settled sessions, not a fleet's entire registered size.

F is **gross indexed Supplier distributions ÷ equal share of the same sessions' Supplier pool**. F=1 means equal share; F=1.2 means 20% above that reference. It is not profit, uptime or an automatic prediction. Predictions require enough relevant earlier/later evidence, assessed separately during product integration.


In [ ]:
DIMENSION = "owner"       # "owner", "domain", "supplier"
SERVICE = None             # None combines collected services; or "eth", "base", etc.
EVIDENCE = None
RECEIPT = None
ANALYSIS = None
RECEIPT_PATH = None
if MODE == "offline":
    RECEIPT_PATH = INCLUDED_PILOT
    print("INCLUDED REVIEWED PILOT: Ethereum, seven days. No fresh collection performed.")
elif RESULT and RESULT.get("receiptPath"):
    RECEIPT_PATH = Path(RESULT["receiptPath"])
if RECEIPT_PATH is not None:
    receipt_mode = "7d" if MODE == "offline" else MODE
    RECEIPT = staged.verify_receipt(RECEIPT_PATH, receipt_mode)
    EVIDENCE = read_json(RECEIPT_PATH.parent/RECEIPT["artifact"])
    ANALYSIS = analyze(EVIDENCE, DIMENSION, SERVICE)
    print(json.dumps({k: RECEIPT[k] for k in ("status", "config", "coverage", "overlap")}, indent=2))
    print(ANALYSIS["notice"])
    import html
    columns = ["label", "actualPokt", "equalSharePokt", "fScore", "differencePercent", "selectedSupplierIdentities", "informativeDays"]
    headers = ["Historical group", "Gross POKT", "Equal-share POKT", "F", "vs equal share (%)", "Selected Supplier IDs", "Informative days"]
    def shown(value):
        return "—" if value is None else f"{value:,.3f}" if isinstance(value, float) else str(value)
    table = '<div style="overflow:auto"><table><thead><tr>' + ''.join('<th>'+h+'</th>' for h in headers) + '</tr></thead><tbody>'
    for row in ANALYSIS["rows"][:20]:
        table += '<tr>'+''.join('<td>'+html.escape(shown(row[c]))+'</td>' for c in columns)+'</tr>'
    table += '</tbody></table></div>'
    try:
        from IPython.display import HTML, display
        display(HTML(table))
    except ImportError:
        for row in ANALYSIS["rows"][:20]:
            print({c: row[c] for c in columns})
    print("Showing up to 20 groups. The JSON export keeps every group. This proof analysis assigns no ranks.")
else:
    print("No successfully reconciled receipt for the selected mode. An investigation alone is not a collection.")


## 8. Save the result and the review package

This writes a clear readiness report, complete group analysis, lossless compact product and a local review ZIP. Packing checks a full round trip before saving; no observation, historical configuration or amount is dropped. The ZIP includes readable code, reference, completed stage receipts/evidence and the investigation/error report. It excludes raw HTTP responses and the large SQLite databases.

Save the notebook with **File → Save Notebook**. Attach the saved notebook and `review-results.zip` **in this conversation**. Nothing is sent to Ramiro or deployed. We will review the source question and data, then update the website and assess fleet-specific historical tests before public GitHub/Vercel setup.


In [ ]:
import zipfile

readiness = staged.readiness(RECEIPT, DIAGNOSIS)
readiness.update(mode=MODE, lastError=LAST_ERROR, sourceCommit=metadata["sourceCommit"])
atomic_json(WORK_DIR/"readiness-report.json", readiness)
atomic_json(WORK_DIR/"source-provenance.json", {"sourceCommit": metadata["sourceCommit"], "packageId": PACKAGE_ID,
             "modules": metadata["modules"], "assetHashes": {k:v["sha256"] for k,v in metadata["assets"].items()}})
print(json.dumps(readiness, indent=2))
if ANALYSIS is not None:
    atomic_json(WORK_DIR/"recorded-earnings-analysis.json", ANALYSIS)
if EVIDENCE is not None:
    packed = export_product(EVIDENCE, WORK_DIR/"indexed-settlements-product.json.gz")
    print("Lossless product:", packed["path"], "·", packed["bytes"], "bytes · exact round trip passed")
if MAKE_REVIEW_ZIP:
    paths = {relative: WORK_DIR/relative for relative in metadata["modules"]}
    paths.update({relative: WORK_DIR/relative for relative in metadata["assets"]})
    for name in ("readiness-report.json", "source-provenance.json", "last-error.json", "recorded-earnings-analysis.json", "indexed-settlements-product.json.gz"):
        if name == "last-error.json" and LAST_ERROR is None:
            continue
        if name in {"recorded-earnings-analysis.json", "indexed-settlements-product.json.gz"} and EVIDENCE is None:
            continue
        if (WORK_DIR/name).is_file():
            paths[name] = WORK_DIR/name
    if diagnosis_path.is_file():
        paths["investigation/result.json"] = diagnosis_path
    for mode in ("7d", "14d", "30d"):
        directory = staged.run_dir(CACHE_ROOT, mode)
        for name in ("run.json", "result.json", "settlement-evidence.json.gz"):
            if (directory/name).is_file():
                paths[f"stages/{mode}/{name}"] = directory/name
    review_zip = WORK_DIR/"review-results.zip"
    temporary = review_zip.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for name, path in sorted(paths.items()):
            archive.write(path, name)
    temporary.replace(review_zip)
    print("REVIEW ZIP:", review_zip)
    print("Size:", f"{review_zip.stat().st_size/1048576:.1f} MiB", "· raw HTTP and SQLite excluded")
if LAST_ERROR:
    print("NEXT: save the notebook and share its review ZIP; keep the current stage and cache.")
elif MODE in {"7d", "investigate"}:
    print("NEXT:", '14d is permitted for indexed-settlement collection.' if DIAGNOSIS and DIAGNOSIS["technicalStatus"] == "passed" else 'Share the investigation report before expansion.')
elif MODE == "14d" and RECEIPT:
    print('NEXT: MODE="30d" may now collect the fixed full month.')
elif MODE == "30d" and RECEIPT:
    print("NEXT: share this notebook and review ZIP here for data review and product integration.")
else:
    print('NEXT: save the notebook, set MODE="7d", and run all cells.')


## Source appendices

These are the exact readable modules used by the executable cells. No hidden library supplies the collection logic. The original two collectors remain unchanged; `staged_collection.py` adds the paired investigation, 14-day orchestration, stage gates and progress heartbeat. The tiny on-chain fixture contains exact selected events from a previously recorded block and is only a parser test, rather than complete live source evidence.


### collector/pokt_live_collector.py

```python
#!/usr/bin/env python3
"""Pocket fleet analysis: reproducible, read-only live collection (Python 3.10+).

No third-party packages, keys or wallet required. This is the NEW Phase 2
collector; original_v5_core.py is the unmodified collection code from V5.

  python pokt_live_collector.py discover --out ./cache
  python pokt_live_collector.py collect --out ./cache --days 30

discover records ALL service metadata and seven mature days of indexed settled
estimated compute units. collect deep-collects services chosen before looking at
fleet earnings, at 03/09/15/21 UTC, for EVERY historically registered Application.
These are sampled sessions, never a monthly earnings census. Missing claims are
unknown, never silently verified zero. Owner-address cohorts are not businesses.

Every response is cached with request, archive height, served height, timestamp,
response hash and wire bytes. Registry snapshots are fetched ONCE per height,
shared across services. Fixed indexer cutoff, cursor checks, single-flight cache,
bounded concurrency, retry/backoff and atomic publication support resumability.
Only a fully successful collection publishes snapshot.json.gz. Failures preserve
completed requests; rerun the EXACT same run.json with --resume.
"""
from __future__ import annotations

import argparse
import concurrent.futures as cf
import gzip
import hashlib
import ipaddress
import json
import os
import re
import random
import socket
import sys
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from decimal import Decimal
from functools import lru_cache
from pathlib import Path

from session_catalog import (
    PARAMS_QUERY, ROSTER_ERA_HEIGHT, block_times, config_at, param_at, roster_page, session_claims,
    staked_suppliers, supplier_configs,
)

VERSION = "phase2-1.2"
REGISTRY_SCHEMA_VERSION = 2
PSL_SHA256 = "ba7d836c0ea57a8bcf9f1fcb868066a7554bed029b441ecc6204ad8494a32612"
RESUMABLE_VERSIONS = {"phase2-1.0", "phase2-1.1", VERSION}
CACHE_NAMESPACE = "phase2-1.0"  # Immutable request cache remains reusable across code fixes.
API = "https://sauron-api.infra.pocket.network"
GRAPHQL = "https://data.pocket.network/graphql"
RPC = "https://sauron-rpc.infra.pocket.network"
PAYOUT_REASONS = ["TLM_RELAY_BURN_EQUALS_MINT_SUPPLIER_SHAREHOLDER_RD",
                  "TLM_GLOBAL_MINT_SUPPLIER_SHAREHOLDER_REWARD_DISTRIBUTION"]
HOURS = [3, 9, 15, 21]
FIXED_SERVICES = ["arb-one", "base", "bsc", "eth", "opbnb", "poly", "robinhood", "solana"]


def utcnow():
    return datetime.now(timezone.utc)


def iso(t):
    return t.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


def dt(value):
    d = datetime.fromisoformat(value.replace("Z", "+00:00"))
    return d.replace(tzinfo=timezone.utc) if d.tzinfo is None else d.astimezone(timezone.utc)


def digest(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f".{os.getpid()}.{threading.get_ident()}.tmp")
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(tmp, "wt", encoding="utf-8") as f:
        json.dump(value, f, separators=(",", ":"), ensure_ascii=False)
    os.replace(tmp, path)


def read_json(path):
    path = Path(path)
    with (gzip.open if path.suffix == ".gz" else open)(path, "rt", encoding="utf-8") as f:
        return json.load(f)


def integer(value, label="amount"):
    n = Decimal(str(value))
    if not n.is_finite() or n < 0 or n != n.to_integral_value():
        raise ValueError(f"Invalid nonnegative integer {label}: {value}")
    return int(n)


@lru_cache(maxsize=1)
def suffix_rules():
    """Pinned Public Suffix List, including private suffixes; no network lookup."""
    path = Path(__file__).with_name("public_suffix_list.dat")
    raw = path.read_bytes()
    if hashlib.sha256(raw).hexdigest() != PSL_SHA256:
        raise ValueError("Public Suffix List checksum mismatch; update the reviewed pin explicitly")
    rules = set()
    for line in raw.decode("utf-8").splitlines():
        rule = line.split("//", 1)[0].strip()
        if rule:
            prefix = "!" if rule.startswith("!") else "*." if rule.startswith("*.") else ""
            name = rule[len(prefix):].encode("idna").decode("ascii").lower()
            rules.add(prefix + name)
    return rules, hashlib.sha256(raw).hexdigest()


@lru_cache(maxsize=32768)
def domain_parts(host):
    """PSL prevailing-rule algorithm; IPs and single labels have no domain."""
    try:
        ipaddress.ip_address(host)
        return {"hostKind": "ip", "publicSuffix": None, "registrableDomain": None,
                "subdomain": None, "subdomainLabels": []}
    except ValueError:
        pass
    labels = host.split(".")
    if len(labels) < 2:
        return {"hostKind": "single_label", "publicSuffix": None, "registrableDomain": None,
                "subdomain": None, "subdomainLabels": []}
    rules, _ = suffix_rules()
    length = 1  # PSL default rule '*'.
    exceptions = []
    for i in range(len(labels)):
        candidate = ".".join(labels[i:])
        if "!" + candidate in rules:
            exceptions.append(len(labels)-i-1)
        if candidate in rules:
            length = max(length, len(labels)-i)
        if i and "*."+candidate in rules:
            length = max(length, len(labels)-i+1)
    if exceptions:
        length = max(exceptions)
    suffix = ".".join(labels[-length:])
    domain = ".".join(labels[-length-1:]) if len(labels)>length else None
    sublabels = labels[:-length-1] if domain else []
    return {"hostKind": "dns", "publicSuffix": suffix, "registrableDomain": domain,
            "subdomain": ".".join(sublabels), "subdomainLabels": sublabels}


def parse_endpoint(endpoint, service_id):
    """Preserve structural facts, never infer service, business or geography.

    Raw URLs stay in the hashed source response. Public fields omit credentials,
    query values and fragments; the source URL hash preserves their provenance.
    """
    raw = str(endpoint.get("url", ""))
    rpc_type = endpoint.get("rpc_type")
    source_hash = hashlib.sha256(raw.encode()).hexdigest()
    result = {"id": "endpoint:"+digest([service_id, source_hash, rpc_type])[:24],
              "serviceId": service_id, "rpcType": rpc_type, "sourceUrlSha256": source_hash}
    try:
        if any(c.isspace() or ord(c)<32 for c in raw):
            raise ValueError("Endpoint URL contains whitespace or control characters")
        url = urllib.parse.urlsplit(raw)
        if url.scheme.lower() not in ("http", "https", "ws", "wss", "grpc", "grpcs") or not url.hostname:
            raise ValueError("Expected an absolute endpoint URL with a supported transport")
        host = url.hostname.lower().rstrip(".").encode("idna").decode("ascii")
        port = url.port  # Raises on invalid or out-of-range ports.
        if not host or any(c.isspace() for c in host) or any(c in host for c in "/\\?#@"):
            raise ValueError("Invalid endpoint hostname")
        try:
            ip = ipaddress.ip_address(host)
            host = str(ip)
        except ValueError:
            if len(host)>253 or any(not re.fullmatch(r"[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?", label) for label in host.split(".")):
                raise ValueError("Invalid DNS hostname")
        authority = "["+host+"]" if ":" in host else host
        if port is not None:
            authority += ":"+str(port)
        redacted = [name for name, present in (("userinfo", url.username is not None or url.password is not None),
                     ("query_values", bool(url.query)), ("fragment", bool(url.fragment))) if present]
        parts=domain_parts(host)
        result.update(parseStatus="parsed", scheme=url.scheme.lower(), hostname=host, port=port,
                      effectivePort=port if port is not None else {"http":80,"https":443,"ws":80,"wss":443}.get(url.scheme.lower()),
                      path=url.path, pathSegments=(url.path.split("/")[1:] if url.path.startswith("/") else url.path.split("/")) if url.path else [],
                      queryKeys=sorted({key for key, _ in urllib.parse.parse_qsl(url.query, keep_blank_values=True)}),
                      redactedParts=redacted, url=urllib.parse.urlunsplit((url.scheme.lower(),authority,url.path,"","")),
                      subdomainTokenGroups=[label.split("-") for label in parts["subdomainLabels"]], **parts)
    except (ValueError, UnicodeError) as exc:
        result.update(parseStatus="invalid", reason=str(exc), hostname=None, registrableDomain=None)
    return result


@lru_cache(maxsize=32768)
def address_parts(address):
    """Validate a Pocket Bech32 account. The payload is opaque, not an actor ID."""
    alphabet = "qpzry9x8gf2tvdw0s3jn54khce6mua7l"
    if (not isinstance(address,str) or len(address)>90 or
            (address.lower()!=address and address.upper()!=address)):
        raise ValueError("Invalid Bech32 account address")
    text=address.lower()
    split=text.rfind("1")
    if split<1 or text[:split]!="pokt" or len(text)-split-1<6:
        raise ValueError("Expected a Pocket account address")
    try:
        values=[alphabet.index(c) for c in text[split+1:]]
    except ValueError as exc:
        raise ValueError("Invalid Bech32 account alphabet") from exc
    prefix=text[:split]
    expanded=[ord(c)>>5 for c in prefix]+[0]+[ord(c)&31 for c in prefix]
    chk=1
    for v in expanded+values:
        top=chk>>25;chk=((chk&0x1ffffff)<<5)^v
        for i,g in enumerate((0x3b6a57b2,0x26508e6d,0x1ea119fa,0x3d4233dd,0x2a1462b3)):
            if (top>>i)&1:chk^=g
    if chk!=1:
        raise ValueError("Pocket account checksum mismatch")
    acc=bits=0;payload=[]
    for value in values[:-6]:
        acc=(acc<<5)|value;bits+=5
        if bits>=8:
            bits-=8;payload.append((acc>>bits)&255)
    if bits>=5 or ((acc<<(8-bits))&255) or len(payload)!=20:
        raise ValueError("Invalid Pocket account payload or padding")
    return {"encoding":"bech32", "prefix":prefix, "payloadBytes":len(payload),
            "checksumValid":True, "payloadHex":bytes(payload).hex()}


def normalized_registry(suppliers, service_ids):
    registry = []
    for supplier in suppliers:
        configs = [c for c in supplier.get("services", []) if c["service_id"] in service_ids]
        if not configs:
            continue
        endpoints = {e["id"]: e for config in configs for e in
                     (parse_endpoint(endpoint, config["service_id"]) for endpoint in config.get("endpoints", []))}
        endpoints = sorted(endpoints.values(), key=lambda e:e["id"])
        registry.append({"id": supplier["operator_address"], "owner": supplier["owner_address"],
                         "operatorAddressParts": address_parts(supplier["operator_address"]),
                         "ownerAddressParts": address_parts(supplier["owner_address"]),
                         "services": sorted({c["service_id"] for c in configs}),
                         "hosts": sorted({e["hostname"] for e in endpoints if e["hostname"]}), "endpoints": endpoints})
    return registry


def upgrade_point_registry(part, client, service_ids):
    """Recover legacy URL structure at the original height; reuse hashed cache.

    An offline export supplies a cache-only Client. A resumed collector may fetch
    a missing raw response at the SAME historical height, never today's registry.
    """
    if part.get("registrySchemaVersion") == REGISTRY_SCHEMA_VERSION:
        return part
    suppliers = client.pages("/pokt-network/poktroll/supplier/supplier", "supplier",
                             part["point"]["height"], limit=10000, dehydrated="true")
    registry = normalized_registry(suppliers, service_ids)
    before = {r["id"]: (r["owner"], sorted(r["services"])) for r in part["registry"]}
    after = {r["id"]: (r["owner"], sorted(r["services"])) for r in registry}
    if before != after:
        raise ValueError("Legacy point disagrees with its pinned raw registry")
    part.update(registry=registry, registrySchemaVersion=REGISTRY_SCHEMA_VERSION)
    return part


class Client:
    def __init__(self, root, workers=4):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.guard = threading.Lock()
        self.locks = {}
        self.metrics = []
        self.workers = workers

    def keylock(self, key):
        with self.guard:
            return self.locks.setdefault(key, threading.Lock())

    def request(self, url, body=None, height=None, label="http", fresh=False):
        key = digest([CACHE_NAMESPACE, url, body, height, iso(utcnow()) if fresh else None])
        path = self.root / "responses" / key[:2] / (key + ".json.gz")
        with self.keylock(key):
            if path.exists():
                cached = read_json(path)
                if digest(cached["response"]) != cached["response_sha256"]:
                    raise ValueError("Cached response checksum mismatch")
                return cached["response"]
            headers = {"Content-Type": "application/json", "Accept-Encoding": "gzip",
                       "User-Agent": "pokt-fleet-analysis/" + VERSION}
            if height is not None:
                headers["x-cosmos-block-height"] = str(height)
            encoded = None if body is None else json.dumps(body).encode()
            for attempt in range(4):
                started, count, success = time.perf_counter(), 0, False
                try:
                    with urllib.request.urlopen(urllib.request.Request(url, encoded, headers), timeout=90) as response:
                        raw = response.read()
                        count = len(raw)
                        served = response.headers.get("x-cosmos-block-height")
                        content_encoding = response.headers.get("Content-Encoding")
                    if height is not None and served != str(height):
                        raise ValueError(f"Archive did not confirm height {height}: {served}")
                    if content_encoding == "gzip":
                        raw = gzip.decompress(raw)
                    value = json.loads(raw)
                    if value.get("errors"):
                        raise ValueError("GraphQL errors: " + json.dumps(value["errors"])[:900])
                    atomic_json(path, {"url": url, "body": body, "requested_height": height,
                                      "served_height": served, "fetched_at": iso(utcnow()),
                                      "received_bytes": count, "response_sha256": digest(value),
                                      "response": value})
                    success = True
                    return value
                except urllib.error.HTTPError as e:
                    if e.code not in (429, 500, 502, 503, 504) or attempt == 3:
                        raise RuntimeError(f"HTTP {e.code}: {e.read().decode(errors='replace')[:900]}") from e
                    delay = e.headers.get("Retry-After")
                    delay = min(30, int(delay)) if delay and delay.isdigit() else 2 ** attempt
                except (urllib.error.URLError, TimeoutError, socket.timeout, ConnectionError):
                    if attempt == 3:
                        raise
                    delay = 2 ** attempt
                finally:
                    metric = {"label": label, "url": url.split("?")[0], "height": height,
                              "seconds": round(time.perf_counter() - started, 3), "bytes": count,
                              "success": success, "finished_at": iso(utcnow())}
                    with self.guard:
                        self.metrics.append(metric)
                        with (self.root / "http_metrics.jsonl").open("a") as f:
                            f.write(json.dumps(metric) + "\n")
                time.sleep(delay)

    def rest(self, path, height=None, fresh=False, **params):
        url = API + path + ("?" + urllib.parse.urlencode(params) if params else "")
        return self.request(url, height=height, label=path.rsplit("/", 1)[-1], fresh=fresh)

    def gql(self, query, fresh=False, **variables):
        value = self.request(GRAPHQL, {"query": query, "variables": variables}, label="graphql", fresh=fresh)
        if value.get("data") is None:
            raise ValueError("GraphQL returned no data")
        return value["data"]

    def pages(self, path, key, height=None, limit=1000, **params):
        records, cursor, seen, identities = [], None, set(), set()
        for page in range(1000):
            p = {**params, "pagination.limit": limit}
            if cursor:
                p["pagination.key"] = cursor
            result = self.rest(path, height, **p)
            for row in result[key]:
                identity = row.get("operator_address", row.get("address", row.get("id")))
                if not identity or identity in identities:
                    raise ValueError("Duplicate or missing REST registry identity")
                identities.add(identity)
                records.append(row)
            cursor = result.get("pagination", {}).get("next_key")
            if not cursor:
                return records
            if cursor in seen:
                raise ValueError("Repeated REST cursor")
            seen.add(cursor)
        raise ValueError("REST pagination failed to finish")

    def parallel(self, fn, items):
        results = []
        with cf.ThreadPoolExecutor(max_workers=self.workers) as pool:
            tasks = {pool.submit(fn, item): item for item in items}
            for i, future in enumerate(cf.as_completed(tasks), 1):
                results.append(future.result())
                if i % 10 == 0 or i == len(tasks):
                    print(f"Completed {i}/{len(tasks)}", flush=True)
        return results


def indexed_services(client, height):
    """Every registered service at one height, from the indexer's Service entity."""
    rows, total = [], None
    for offset in range(0, 100000, 100):
        conn = client.gql('query($h:String,$o:Int){services(first:100,offset:$o,blockHeight:$h,orderBy:ID_ASC)'
                          '{totalCount nodes{id name computeUnitsPerRelay}}}', h=str(height), o=offset)["services"]
        total = conn["totalCount"] if total is None else total
        if conn["totalCount"] != total:
            raise ValueError("Service count changed while paginating")
        rows.extend({"id": s["id"], "name": s["name"], "compute_units_per_relay": s["computeUnitsPerRelay"]} for s in conn["nodes"])
        if len(rows) >= total or not conn["nodes"]:
            break
    if len(rows) != total or len({s["id"] for s in rows}) != total:
        raise ValueError("Incomplete or repeated service catalogue")
    return rows


def discover(client, end=None, max_services=8, target=.95, cutoff=None):
    indexed = client.gql('{_metadata{lastProcessedHeight chain} blocks(first:1,orderBy:ID_DESC){nodes{id timestamp}}}', fresh=True)
    meta = indexed["_metadata"]
    if meta["chain"] != "pocket":
        raise ValueError("Expected Pocket indexer")
    latest = min(int(meta["lastProcessedHeight"]), int(indexed["blocks"]["nodes"][0]["id"]))
    # A fixed historical cutoff reproduces an earlier collection: services as registered then.
    if cutoff is not None and not 0 < cutoff <= latest:
        raise ValueError(f"Cutoff must be an indexed height no later than {latest}")
    cutoff = latest if cutoff is None else cutoff
    head = {"chain_id": meta["chain"], "height": str(latest), "time": indexed["blocks"]["nodes"][0]["timestamp"]}
    head_time = dt(head["time"])
    safe_end = head_time.replace(hour=0, minute=0, second=0, microsecond=0) - timedelta(days=2)
    end_time = dt(end) if end else safe_end
    if end_time > safe_end or end_time != end_time.replace(hour=0, minute=0, second=0, microsecond=0):
        raise ValueError(f"Use a whole UTC end date no later than {safe_end.date()}")
    service_rows = indexed_services(client, cutoff)
    services = [{"id": s["id"], "name": s.get("name") or s["id"],
                 "computeUnitsPerRelay": s.get("compute_units_per_relay")} for s in service_rows]
    # Explicit service list prevents top-by-settled from silently omitting services.
    query = '''query($ids:[String],$a:Datetime,$b:Datetime){getServiceUsageJson(
        services:$ids,rangeStart:$a,rangeEnd:$b,bucket:"day",topBySettled:200,
        byService:true,fillEmptyBuckets:false)}'''
    totals = defaultdict(lambda: {"estimatedComputeUnits": 0, "estimatedRelays": 0, "claims": 0, "coverage": []})
    for offset in range(0, len(services), 100):
        result = client.gql(query, ids=[s["id"] for s in services[offset:offset+100]],
                            a=iso(end_time-timedelta(days=7)), b=iso(end_time))
        usage = result["getServiceUsageJson"]
        rows = usage["data"]
        coverage = usage["range"]
        if coverage.get("gaps") or not coverage.get("covered_from") or dt(coverage["covered_from"]) > end_time-timedelta(days=7) or dt(coverage["covered_to"]) < end_time:
            raise ValueError("Indexer activity window has gaps or incomplete coverage")
        for row in rows:
            target_row = totals[row["service_id"]]
            for k in ("estimatedComputeUnits", "estimatedRelays", "claims"):
                source_k = {"estimatedComputeUnits": "estimated_compute_units", "estimatedRelays": "estimated_relays", "claims": "claims"}[k]
                if row[source_k] is None:
                    raise ValueError(f"Missing indexed activity field {k}")
                target_row[k] += integer(row[source_k], k)
            if not target_row["coverage"]:
                target_row["coverage"].append(coverage)
    for s in services:
        s.update(totals.get(s["id"], {"estimatedComputeUnits": None, "estimatedRelays": None, "claims": None, "coverage": []}))
    ranked = sorted((s for s in services if s["estimatedComputeUnits"]), key=lambda s: (-s["estimatedComputeUnits"], s["id"]))
    total = sum(s["estimatedComputeUnits"] for s in ranked)
    chosen, cumulative = [], 0
    for s in ranked:
        if len(chosen) >= max_services or (chosen and cumulative / total >= target):
            break
        chosen.append(s["id"])
        cumulative += s["estimatedComputeUnits"]
    if not chosen:
        raise ValueError("No indexed settled activity available for service selection")
    catalog = {"collectorVersion": VERSION, "fetchedAt": iso(utcnow()), "chainHead": head,
               "indexerMetadata": meta, "cutoff": cutoff, "endExclusive": iso(end_time),
               "activityWindow": {"start": iso(end_time-timedelta(days=7)), "endExclusive": iso(end_time)},
               "services": services, "selectedServices": chosen,
               "activityShare": cumulative/total, "targetShare": target, "maxServices": max_services,
               "selectionRule": "Top indexed settled estimated compute units; chosen before fleet earnings. Not all request traffic."}
    atomic_json(client.root / "catalog.json", catalog)
    print(json.dumps({"services": len(services), "chosen": chosen, "activityShare": cumulative/total,
                      "endExclusive": iso(end_time), "cutoff": cutoff}), flush=True)
    return catalog


def resolve_schedule(client, start, end, hours):
    times = []
    for day in range((dt(end)-dt(start)).days):
        times.extend(iso(dt(start)+timedelta(days=day, hours=h)) for h in hours)
    schedule = []
    # The indexer accepts at most 20 aliases per request.
    for offset in range(0, len(times), 20):
        part = times[offset:offset+20]
        fields = [f'b{i}:blocks(first:1,orderBy:ID_ASC,filter:{{timestamp:{{greaterThanOrEqualTo:{json.dumps(t)}}}}}){{nodes{{id timestamp}}}}' for i, t in enumerate(part)]
        result = client.gql("{" + " ".join(fields) + "}")
        for i, t in enumerate(part):
            rows = result[f"b{i}"]["nodes"]
            if len(rows) != 1 or not timedelta(0) <= dt(rows[0]["timestamp"])-dt(t) < timedelta(minutes=10):
                raise ValueError("Indexer cannot resolve sample time to a block within ten minutes")
            schedule.append({"height": int(rows[0]["id"]), "observedAt": iso(dt(rows[0]["timestamp"])), "scheduledAt": t})
    return schedule


def collect_point(client, point, service_ids, history, cutoff):
    """One sample height from the pocketdex session catalog.

    The sessions covering the height, their rosters and outcomes come from
    getSessionRostersJson. A stake or config change takes effect at the next
    session start, so eligible populations, owners and endpoints are the state
    staked before the start of the session covering the height (start - 1),
    which is the registry state the archive returned at the height. The authority
    for N is the session: its candidates must equal the population.
    """
    height = point["height"]
    shared = {r["key"]: str(param_at(history, "shared", r["key"], height)) for r in history if r["namespace"] == "shared"}
    start_time = iso(dt(point["observedAt"])-timedelta(hours=1))
    rows, before = [], None
    for _ in range(100):
        page = roster_page(client, start_time, iso(dt(point["observedAt"])+timedelta(hours=2)), service_ids, before=before)
        rows.extend(page)
        if len(page) < 1000:
            break
        before = (int(page[-1]["start_height"]), page[-1]["session_id"])
    covering = [r for r in rows if int(r["start_height"]) <= height <= int(r["end_height"])]
    if not rows or int(rows[-1]["start_height"]) >= height-int(shared["num_blocks_per_session"]):
        raise ValueError("Session catalog start range does not reach before the sample height")
    starts = {int(r["start_height"]) for r in covering}
    if len(starts) != 1:
        raise ValueError(f"Sessions covering height {height} do not share one start")
    start = starts.pop()
    statuses = None
    suppliers = staked_suppliers(client, start-1)
    def eligible_for(service_id):
        return {s["supplierId"] for s in suppliers if service_id in s["serviceIds"]}
    candidates = {(r["service_id"], int(r["candidates"])) for r in covering}
    if any(n != len(eligible_for(s)) for s, n in candidates):
        # An Unstaking supplier can still be selected until its unbonding ends.
        statuses = ["Staked", "Unstaking"]
        suppliers = staked_suppliers(client, start-1, statuses)
        if any(n != len(eligible_for(s)) for s, n in candidates):
            raise ValueError(f"Eligible population disagrees with the session candidates at height {height}")
    populations, sessions, claims = [], [], []
    for service_id in service_ids:
        eligible_ids = eligible_for(service_id)
        service_rows = sorted((r for r in covering if r["service_id"] == service_id), key=lambda r: r["application_id"])
        cap = param_at(history, "session", "num_suppliers_per_session", start)
        population = {"id": f"{service_id}:{height}", "serviceId": service_id, **point,
                      "totalEligible": len(eligible_ids), "selectionCap": cap,
                      "eligibleSupplierIds": sorted(eligible_ids)}
        populations.append(population)
        for r in service_rows:
            selected = r["member_ids"]
            if len(set(selected)) != len(selected) or not set(selected).issubset(eligible_ids):
                raise ValueError("Session roster duplicates or not in pinned eligible population")
            if len(selected) != min(cap, len(eligible_ids)) or int(r["roster_size"]) != len(selected):
                raise ValueError("Returned roster length disagrees with historical cap and population")
            sessions.append({"id": r["session_id"], "serviceId": service_id,
                             "applicationId": r["application_id"], "gatewayId": None,
                             "populationId": population["id"], "startedAt": point["observedAt"],
                             "scheduledAt": point["scheduledAt"], "sampleHeight": height,
                             "sessionStartHeight": int(r["start_height"]),
                             "sessionEndHeight": int(r["end_height"]),
                             "rosterSource": "session_catalog",
                             "selectedSupplierIds": selected, "selectedCount": len(selected), "rosterStatus": "complete"})
            claims.extend({**c, "numEstimatedRelays": None} for c in session_claims(r, {"cutoff": cutoff}, service_id))
    staked = {s["supplierId"]: s for s in suppliers if any(sid in s["serviceIds"] for sid in service_ids)}
    return {"point": point, "populations": populations, "sessions": sessions, "claims": claims,
            "stakedSuppliers": [{"id": sid, "owner": s["ownerId"], "services": sorted(set(s["serviceIds"]) & set(service_ids))}
                                for sid, s in sorted(staked.items())],
            "eligibilityStakeStatuses": statuses or ["Staked"], "sharedParams": shared, "sessionStartHeight": start,
            "claimedMembers": {r["session_id"]: int(r["claimed"]) for r in covering}}


def point_registry(part, configs, service_ids):
    """Registry at the sample height: owners and the endpoints declared before its session start."""
    start = part["sessionStartHeight"]
    raw = [{"operator_address": s["id"], "owner_address": s["owner"],
            "services": [{"service_id": sid, "endpoints": config_at(configs, s["id"], sid, start)} for sid in s["services"]]}
           for s in part["stakedSuppliers"]]
    part.update(registry=normalized_registry(raw, service_ids),
                registrySchemaVersion=REGISTRY_SCHEMA_VERSION)
    return part


def assemble(collected, claims, catalog, config, metrics):
    # Point files are streamed, not retained as 120 decoded registry dictionaries.
    collected.sort(key=lambda r: r['point']['height'] if isinstance(r,dict) else int(Path(r).name.split('.')[0]))
    suppliers, populations, sessions, endpoint_catalog = {}, [], {}, {}
    for item in collected:
        part=item if isinstance(item,dict) else read_json(item)
        point_date = part["point"]["observedAt"][:10]
        point_time = part["point"]["observedAt"]
        # Reuse one canonical address string across all populations and rosters.
        for pop in part['populations']:
            pop['eligibleSupplierIds']=[sys.intern(sid) for sid in pop['eligibleSupplierIds']]
        populations.extend(part["populations"])
        for r in part["registry"]:
            r['id']=sys.intern(r['id']);r['owner']=sys.intern(r['owner'])
            s = suppliers.setdefault(r["id"], {"id": r["id"], "name": short_address(r["id"]),
                    "fleetId": "", "groupId": "unverified", "groupName": "Infrastructure not attributed",
                    "serviceIds": set(), "endpointHosts": set(), "endpointIds": set(), "ownerHistory": [], "endpointHistory": []})
            s["serviceIds"].update(r["services"])
            s["endpointHosts"].update(r["hosts"])
            endpoint_ids = sorted(e["id"] for e in r.get("endpoints", []))
            for endpoint in r.get("endpoints", []):
                if endpoint["id"] in endpoint_catalog and endpoint_catalog[endpoint["id"]] != endpoint:
                    raise ValueError("Endpoint identity collision")
                endpoint_catalog.setdefault(endpoint["id"], endpoint)
            s["endpointIds"].update(endpoint_ids)
            if r.get("operatorAddressParts"):
                s["operatorAddressParts"] = r["operatorAddressParts"]
            if (not s["endpointHistory"] or s["endpointHistory"][-1]["hosts"] != r["hosts"]
                    or s["endpointHistory"][-1].get("endpointIds", []) != endpoint_ids):
                if s["endpointHistory"]:
                    s["endpointHistory"][-1]["end"] = point_time
                history = {"hosts": r["hosts"], "start": point_time, "end": None,
                           "firstObservedHeight": part["point"]["height"], "lastObservedHeight": part["point"]["height"]}
                if "endpoints" in r:
                    history["endpointIds"] = endpoint_ids
                s["endpointHistory"].append(history)
            else:
                s["endpointHistory"][-1]["lastObservedHeight"] = part["point"]["height"]
            if not s["ownerHistory"] or s["ownerHistory"][-1]["ownerId"] != r["owner"]:
                if s["ownerHistory"]:
                    s["ownerHistory"][-1]["end"] = point_time
                ownership = {"ownerId": r["owner"], "start": point_time, "end": None}
                if r.get("ownerAddressParts"):
                    ownership["addressParts"] = r["ownerAddressParts"]
                s["ownerHistory"].append(ownership)
        for s in part["sessions"]:
            s['selectedSupplierIds']=[sys.intern(sid) for sid in s['selectedSupplierIds']]
            sessions.setdefault(s["id"], s)
    by_session = defaultdict(list)
    claim_keys = set()
    for c in claims:
        key = (c["sessionId"], c["supplierId"])
        if key in claim_keys:
            raise ValueError("Duplicate settled claim for one Supplier in one session")
        claim_keys.add(key)
        session = sessions[c["sessionId"]]
        if c["supplierId"] not in session["selectedSupplierIds"] or c["applicationId"] != session["applicationId"]:
            raise ValueError("Claim not attributable to returned historical roster")
        by_session[c["sessionId"]].append(c)
    for sid, s in sessions.items():
        cs = by_session[sid]
        complete = bool(cs) and all(c["payoutVerified"] for c in cs)
        s.update(payoutStatus="complete" if complete else "missing",
                 totalPayoutPokt=sum(integer(c["payoutUpokt"]) for c in cs)/1e6 if complete else None,
                 supplierPayouts=[{"supplierId": c["supplierId"], "pokt": integer(c["payoutUpokt"])/1e6} for c in cs] if complete else [],
                 outcomeAvailableAt=max((iso(dt(c["block"]["timestamp"])) for c in cs), default=None),
                 # The session catalog has no per-session estimated relays.
                 settledClaimCount=len(cs), indexedEstimatedRelays=None)
    owners = defaultdict(list)
    for s in suppliers.values():
        s["serviceIds"] = sorted(s["serviceIds"])
        s["endpointHosts"] = sorted(s["endpointHosts"])
        s["endpointIds"] = sorted(s["endpointIds"])
        # Product scopes use historical owner membership, not this display grouping.
        owner = s["ownerHistory"][-1]["ownerId"]
        s["fleetId"] = "owner:" + owner
        for historical_owner in {h["ownerId"] for h in s["ownerHistory"]}:
            owners[historical_owner].append(s)
    fleets = []
    for owner, members in sorted(owners.items()):
        services = sorted({sid for s in members for sid in s["serviceIds"]})
        primary = next((sid for sid in catalog["selectedServices"] if sid in services), services[0])
        fleets.append({"id": "owner:"+owner, "ownerAddress": owner, "name": short_address(owner),
                       "primary": primary, "services": services, "exampleCount": len(members),
                       "identityKind": "owner_address", "tag": "Owner-address group",
                       "note": "Historical on-chain owner membership. Business identity not verified."})
    service_map = {s["id"]: s for s in catalog["services"]}
    live_services = [{"id": sid, "name": service_map[sid]["name"], "shortName": sid.upper()[:10],
                      "selectionCap": populations[-1]["selectionCap"]} for sid in config["services"]]
    indexed_total = sum(s.get('estimatedComputeUnits') or 0 for s in catalog['services'])
    activity_share = (sum(service_map[sid].get('estimatedComputeUnits') or 0 for sid in config['services'])/indexed_total
                      if indexed_total else catalog['activityShare'])
    selection_rule = (catalog['selectionRule'] if set(config['services'])==set(catalog['selectedServices'])
                      else 'Explicit service list; selected before collecting fleet earnings.')
    snapshot = {"schemaVersion": 1, "dataKind": "live", "snapshotId": "live-"+digest(config)[:16],
                "generatedAt": iso(utcnow()), "period": {"start": config["start"], "endExclusive": config["endExclusive"], "timezone": "UTC"},
                "description": "Live mainnet archive and indexer; sampled sessions. Owner-address groups are not verified businesses. No claims means unknown outcome, not verified zero.",
                "sampling": f"{len(config['hours'])} fixed UTC samples/day ({','.join(map(str,config['hours']))}); all historically eligible Applications per service; duplicate sessions removed. Indexed outcomes at block {config['cutoff']}.",
                "networkScopeValue": "live-sample", "networkNotice": "Sampled sessions for the collected services, with complete historical eligible populations. This is not all traffic or monthly total earnings.",
                "services": live_services, "fleets": fleets, "suppliers": list(suppliers.values()), "gateways": [],
                "endpointCatalog": sorted(endpoint_catalog.values(), key=lambda e:e["id"]),
                "applications": [{"id": a, "name": short_address(a), "gatewayId": None} for a in sorted({s['applicationId'] for s in sessions.values()})],
                "populations": populations, "sessions": sorted(sessions.values(), key=lambda s: (s['startedAt'], s['id'])),
                "liveSource": {"api": GRAPHQL, "graphql": GRAPHQL, "cutoff": config["cutoff"], "collectorVersion": VERSION,
                    "config": config, "registeredServiceCount": len(catalog["services"]), "activityShare": activity_share,
                    "catalogFetchedAt": catalog["fetchedAt"], "activityWindow": catalog["activityWindow"],
                    "serviceSelection": {"targetShare": catalog["targetShare"], "maxServices": catalog["maxServices"],
                        "targetReached": activity_share >= catalog["targetShare"], "rule": selection_rule},
                    "selectedServices": config["services"], "screeningSelectedServices": catalog["selectedServices"],
                    "endpointParsing": {"schemaVersion": REGISTRY_SCHEMA_VERSION, "processorVersion": VERSION,
                        "suffixListSource": "https://publicsuffix.org/list/public_suffix_list.dat", "suffixListSha256": PSL_SHA256,
                        "includesPrivateSuffixes": True, "endpointCount": len(endpoint_catalog),
                        "suffixListVersion": "2026-10-07_07-28-19_UTC",
                        "accountValidation": "Pocket Bech32 prefix, checksum, padding and 20-byte account payload; role comes from registry fields.",
                        "invalidEndpointCount": sum(e["parseStatus"] != "parsed" for e in endpoint_catalog.values()),
                        "identityRule": "Recorded service-specific endpoint association; business ownership, location and physical servers are not inferred."},
                    "serviceCatalog": catalog["services"], "collectionMetrics": metrics,
                    "coverage": {"samplePoints": len(collected), "sessions": len(sessions), "settledClaims": len(claims),
                        "comparableSessions": sum(s['payoutStatus']=='complete' for s in sessions.values()),
                        "unverifiedPayoutClaims": sum(not c['payoutVerified'] for c in claims)}}}
    snapshot["liveSource"]["leaderboards"] = build_leaderboards(snapshot)
    snapshot["liveSource"]["domainLeaderboards"] = build_leaderboards(snapshot, "domain")
    return snapshot


def build_leaderboards(snapshot, grouping="owner"):
    """Precompute fair within-service comparisons; never rank sparse/partial rows."""
    def quantile(values, p):
        x = (len(values)-1)*p
        lo = int(x)
        hi = min(lo+1, len(values)-1)
        return values[lo] + (values[hi]-values[lo])*(x-lo)
    suppliers = {s['id']: s for s in snapshot['suppliers']}
    endpoints = {e['id']: e for e in snapshot.get('endpointCatalog', [])}
    populations = {p['id']: p for p in snapshot['populations']}
    owners_by_population = {}
    for pid, pop in populations.items():
        at = pop['observedAt']
        ownership = {sid: next((h['ownerId'] for h in suppliers[sid]['ownerHistory'] if h['start']<=at and (not h['end'] or at<h['end'])), None) for sid in pop['eligibleSupplierIds']}
        if any(owner is None for owner in ownership.values()):
            raise ValueError('Missing owner at historical observation')
        memberships = {}
        counts, group_owners = defaultdict(int), defaultdict(set)
        for sid, owner in ownership.items():
            if grouping == "owner":
                groups = {owner}
            else:
                history = next((h for h in suppliers[sid]['endpointHistory'] if h['start']<=at and (not h['end'] or at<h['end'])), {})
                groups = {endpoints[eid]['registrableDomain'] for eid in history.get('endpointIds', [])
                          if endpoints[eid]['serviceId']==pop['serviceId'] and endpoints[eid].get('registrableDomain')}
            memberships[sid] = groups
            for group in groups:
                counts[group] += 1
                group_owners[group].add(owner)
        owners_by_population[pid] = (memberships, counts, group_owners)
    boards = {}
    for days in (7, 14, 30):
        lower = max(dt(snapshot['period']['start']), dt(snapshot['period']['endExclusive'])-timedelta(days=days))
        board = {}
        for service in snapshot['services']:
            sessions = [s for s in snapshot['sessions'] if s['serviceId']==service['id'] and dt(s['startedAt'])>=lower]
            owners = {o for s in sessions for o in owners_by_population[s['populationId']][1]}
            stats = {o: {'actualPokt': 0, 'equalSharePokt': 0, 'selectedPositions': 0, 'presentSessions': 0, 'daily': defaultdict(lambda:[0.0,0.0]), 'positiveSessions': 0} for o in owners}
            comparable = sum(s['payoutStatus']=='complete' for s in sessions)
            for s in sessions:
                ownership, counts, _ = owners_by_population[s['populationId']]
                selected = defaultdict(int)
                for sid in s['selectedSupplierIds']:
                    for group in ownership[sid]:
                        selected[group] += 1
                for owner, x in selected.items():
                    stats[owner]['selectedPositions'] += x
                    stats[owner]['presentSessions'] += 1
                if s['payoutStatus']!='complete':
                    continue
                for owner_stats in stats.values():
                    owner_stats['daily'][s['startedAt'][:10]]  # Retain days with zero opportunity.
                actual = defaultdict(float)
                for p in s['supplierPayouts']:
                    for group in ownership[p['supplierId']]:
                        actual[group] += p['pokt']
                for owner, x in selected.items():
                    ref = s['totalPayoutPokt']*x/s['selectedCount']
                    r = stats[owner]
                    r['actualPokt'] += actual[owner]
                    r['equalSharePokt'] += ref
                    r['daily'][s['startedAt'][:10]][0] += actual[owner]
                    r['daily'][s['startedAt'][:10]][1] += ref
                    r['positiveSessions'] += ref>0
            latest = next((p for p in reversed(snapshot['populations']) if p['serviceId']==service['id']), None)
            latest_counts = owners_by_population[latest['id']][1] if latest else {}
            rows = []
            for owner, r in stats.items():
                daily = list(r.pop('daily').values())
                informative_days = sum(b>0 for a,b in daily)
                ref = r['equalSharePokt']
                ratio = r['actualPokt']/ref if ref>0 else None
                coverage = comparable/len(sessions) if sessions else 0
                largest = max((b/ref for a,b in daily), default=0) if ref else None
                availability = 'unavailable' if ratio is None else 'partial' if coverage<.9 else 'limited' if informative_days<7 or r['positiveSessions']<24 or largest>.4 else 'available'
                interval = None
                if informative_days>=7 and r['positiveSessions']>=24:
                    rng = random.Random(int(digest([snapshot['snapshotId'],service['id'],days,owner])[:16],16))
                    draws = []
                    for _ in range(2000):
                        sample = [daily[rng.randrange(len(daily))] for _ in daily]
                        den = sum(b for a,b in sample)
                        if den>0:
                            draws.append(sum(a for a,b in sample)/den)
                    if len(draws)>=1800:
                        draws.sort()
                        interval = [quantile(draws,.025),quantile(draws,.975)]
                    elif availability=='available':
                        availability='limited'
                identity = ({'ownerAddress': owner, 'fleetId':'owner:'+owner, 'name':short_address(owner),
                             'scope':{'type':'fleet','value':'owner:'+owner}} if grouping=='owner' else
                            {'domain':owner, 'name':owner, 'scope':{'type':'domain','value':owner},
                             'ownerCount':len(owners_by_population[latest['id']][2].get(owner,set())) if latest else 0})
                rows.append({**r, **identity, 'grouping':grouping,
                             'eligibleSuppliers':latest_counts.get(owner,0), 'ratio':ratio, 'differencePercent':(ratio-1)*100 if ratio is not None else None,
                             'intervalRatio':interval, 'availability':availability, 'informativeDays':informative_days,
                             'coverage':coverage, 'recordedSessions':len(sessions), 'comparableSessions':comparable,
                             'largestDayShare':largest, 'presenceFraction':r['presentSessions']/len(sessions) if sessions else None})
            board[service['id']] = sorted(rows,key=lambda r:(r['availability']!='available',-r['actualPokt'],r.get('domain',r.get('ownerAddress'))))
        boards[str(days)] = board
    return boards


def short_address(address):
    return address[:10] + "…" + address[-6:]


def collect(client, catalog, days=30, service_ids=None, resume=None, hours=HOURS):
    started = time.perf_counter()
    if resume:
        config = read_json(resume)["config"]
        if config["collectorVersion"] not in RESUMABLE_VERSIONS:
            raise ValueError("Resume collector version mismatch")
    else:
        end = dt(catalog["endExclusive"])
        service_ids = service_ids or catalog["selectedServices"]
        available_ids = {s["id"] for s in catalog["services"]}
        if not set(service_ids).issubset(available_ids):
            raise ValueError("Unknown service ID")
        config = {"collectorVersion": VERSION, "start": iso(end-timedelta(days=days)), "endExclusive": iso(end),
                  "services": sorted(set(service_ids)), "hours": hours, "applicationPolicy": "all_historical",
                  "rosterMode":"session_catalog",
                  "cutoff": catalog["cutoff"], "settlementBufferDays": 2}
    run_dir = client.root / "runs" / digest(config)[:16]
    run_dir.mkdir(parents=True, exist_ok=True)
    if resume and (run_dir/'catalog.json').exists():
        catalog=read_json(run_dir/'catalog.json')
    if catalog['cutoff']!=config['cutoff']:
        raise ValueError('Resume requires the original catalogue and fixed indexer cutoff')
    atomic_json(run_dir/'catalog.json',catalog)
    state = {"config": config, "status": "collecting", "startedAt": iso(utcnow())}
    atomic_json(run_dir / "run.json", state)
    try:
        schedule = resolve_schedule(client, config["start"], config["endExclusive"], config["hours"])
        if any(p['height']<ROSTER_ERA_HEIGHT for p in schedule):
            raise ValueError("Collection is restricted to the catalog's session rosters")
        history=client.gql(PARAMS_QUERY,keys=None)['getParamHistoryList']
        print(f"Collecting {len(schedule)} shared historical registries, services {config['services']}, all historical Applications.", flush=True)
        # Sequential: one sample height at a time against the shared public API.
        collected=[]
        for p in schedule:
            path = run_dir / "points" / (str(p["height"])+".json.gz")
            if not path.exists():
                atomic_json(path, collect_point(client, p, config["services"], history, config["cutoff"]))
            collected.append(path)
        # Point files are streamed: one decoded registry at a time. Only points still
        # without a registry need the members' config history.
        pairs=set()
        for path in collected:
            part=read_json(path)
            if part.get("registrySchemaVersion") != REGISTRY_SCHEMA_VERSION:
                pairs.update((s['id'],sid) for s in part['stakedSuppliers'] for sid in s['services'])
        configs=supplier_configs(client,sorted(pairs)) if pairs else {}
        sessions={};claims=[]
        for path in collected:
            part=read_json(path)
            if part.get("registrySchemaVersion") != REGISTRY_SCHEMA_VERSION:
                atomic_json(path, point_registry(part, configs, config["services"]))
            shared=part['sharedParams']
            # Conservative upper guard on timing offsets, in addition to the UTC
            # two-day exclusion. Uses the historical effective settings.
            lag=sum(int(v) for k,v in shared.items() if k.endswith('_offset_blocks'))+2*int(shared['num_blocks_per_session'])
            if any(s['sessionEndHeight']+lag>=config['cutoff'] for s in part['sessions']):
                raise ValueError('An observed session has not passed the conservative historical settlement guard')
            for session in part['sessions']:
                if session['id'] not in sessions:
                    sessions[session['id']]=session
                    claims.extend(c for c in part['claims'] if c['sessionId']==session['id'])
        times=block_times(client,[c['blockId'] for c in claims],config,window=False)
        for c in claims:
            c['block']={'timestamp':times[int(c['blockId'])]}
        print(f"Settled payouts for {len(sessions)} unique sessions from the session catalog.", flush=True)
        metrics = {"secondsThisInvocation": round(time.perf_counter()-started, 2), "httpAttemptsThisInvocation": len(client.metrics),
                   "wireBytesThisInvocation": sum(m['bytes'] for m in client.metrics), "sourceCacheRoot": "responses/"}
        snapshot = assemble(collected, claims, catalog, config, metrics)
        snapshot['liveSource']['rosterVerification']={'source':'getSessionRostersJson','rostersFromHeight':ROSTER_ERA_HEIGHT,
                    'eligibilityStakeStatuses':sorted({s for path in collected for s in read_json(path)['eligibilityStakeStatuses']}),
                    'rule':'Every eligible population equals the candidates of the sessions covering its height.'}
        atomic_json(run_dir / "snapshot.json.gz", snapshot)
        # Same evidence, compact wire representation for a fast static product.
        supplier_index={s['id']:i for i,s in enumerate(snapshot['suppliers'])}
        product={**snapshot,'populations':[],'wireCodec':'supplier-indices-v1'}
        for p in snapshot['populations']:
            product['populations'].append({**{k:v for k,v in p.items() if k!='eligibleSupplierIds'},
                                          'eligibleSupplierIndices':[supplier_index[sid] for sid in p['eligibleSupplierIds']]})
        atomic_json(run_dir/'product-snapshot.json.gz',product)
        atomic_json(run_dir / "claims.json.gz", claims)
        state.update(status="complete", completedAt=iso(utcnow()), metrics=metrics, snapshotPath=str(run_dir/"snapshot.json.gz"))
        atomic_json(run_dir / "run.json", state)
        atomic_json(client.root / "latest.json", state)
        print(json.dumps({"snapshot": state["snapshotPath"], "coverage": snapshot['liveSource']['coverage'], "metrics": metrics}), flush=True)
        return snapshot
    except BaseException as e:
        state.update(status="failed", error=str(e), failedAt=iso(utcnow()))
        atomic_json(run_dir / "run.json", state)
        print(f"Collection incomplete. Cached work is retained. Resume: --resume {run_dir/'run.json'}", flush=True)
        raise


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("command", choices=["discover", "collect"])
    p.add_argument("--out", type=Path, default=Path("pokt-live-cache"))
    p.add_argument("--days", type=int, default=30)
    p.add_argument("--services", help="Comma-separated explicit service IDs; overrides activity-based selection")
    p.add_argument("--fixed-eight", action="store_true", help="Keep the eight reviewed services fixed rather than reselecting by activity")
    p.add_argument("--max-services", type=int, default=8)
    p.add_argument("--activity-share", type=float, default=.95)
    p.add_argument("--end", help="Whole UTC exclusive end date; default excludes two recent UTC days")
    p.add_argument("--workers", type=int, default=2, choices=range(1, 5),help='Two workers limit registry decode memory; up to four when resources permit')
    p.add_argument("--resume", type=Path, help="Exact previous run.json; retains cutoff and sampling frame")
    p.add_argument('--reuse-catalog',action='store_true',help='Start a new run with the saved screening window and cutoff; new collections refresh the catalogue by default')
    args = p.parse_args()
    if not 1 <= args.days <= 90 or not 1 <= args.max_services <= 50 or not 0 < args.activity_share <= 1:
        p.error("Use days 1–90, max-services 1–50, activity-share (0,1]")
    if args.fixed_eight and args.services:
        p.error("Choose --fixed-eight or --services")
    client = Client(args.out, args.workers)
    catalog_path = args.out / "catalog.json"
    if args.resume:
        frozen=args.resume.parent/'catalog.json'
        if frozen.exists():
            catalog_path=frozen
        if not catalog_path.exists():
            p.error('Resume requires the saved original catalogue; do not refresh its cutoff')
    if args.command == "discover" or not catalog_path.exists() or (not args.resume and not args.reuse_catalog):
        catalog = discover(client, args.end, args.max_services, args.activity_share)
    else:
        catalog = read_json(catalog_path)
    if args.command == "collect":
        services = FIXED_SERVICES if args.fixed_eight else [s.strip() for s in args.services.split(",")] if args.services else None
        if args.resume and services and set(services) != set(read_json(args.resume)["config"]["services"]):
            p.error("Resume service list differs from the frozen run")
        collect(client, catalog, args.days, services, args.resume)


if __name__ == "__main__":
    main()

```

### collector/session_catalog.py

```python
"""Read-only access to the pocketdex session catalog (data.pocket.network/graphql).

Python 3.10+, standard library only. Both collectors read rosters, outcomes,
supplier service configs, stake state and governance parameters through these
functions, with the caller's audited caching Client. Numbers arrive as strings.
"""
from __future__ import annotations

from collections import defaultdict
from datetime import datetime, timezone

ROSTERS_QUERY = '''query($a:Datetime,$b:Datetime,$services:[String],$ids:[String],$apps:[String],$h:BigInt,$sid:String){
 getSessionRostersJson(rangeStart:$a,rangeEnd:$b,services:$services,sessionIds:$ids,applications:$apps,rowLimit:1000,
 beforeHeight:$h,beforeSessionId:$sid)}'''
CONFIG_QUERY = '''query($ids:[String],$services:[String]){getSuppliersConfigHistoryJson(suppliers:$ids,services:$services)}'''
NETWORK_QUERY = '''query($a:Datetime,$b:Datetime,$services:[String]){getNetworkSessionsJson(
 rangeStart:$a,rangeEnd:$b,services:$services,byService:true)}'''
PARAMS_QUERY = '''query($keys:[String]){getParamHistoryList(keys:$keys){namespace key height activeAt value}}'''
STAKED_QUERY = '''query($h:BigInt,$after:String,$statuses:[String]){getStakedSuppliersList(
 atHeight:$h,afterSupplierId:$after,stakeStatuses:$statuses,rowLimit:1000){supplierId ownerId serviceIds height}}'''
BLOCKS_QUERY = '''query($filter:BlockFilter!){blocks(first:200,filter:$filter){
 totalCount pageInfo{hasNextPage} nodes{id timestamp}}}'''
PAGE_SIZE = 1000
# The catalog computes rosters as poktroll's session hydrator does from v0.1.33 on.
ROSTER_ERA_HEIGHT = 703870
# Catalog member states with a settled claim (pocketdex sessionRosters: paid > 0, settled_zero = 0).
SETTLED_STATES = ("paid", "settled_zero")
# poktroll proto/pocket/shared/service.proto RPCType; the catalog returns the enum number.
RPC_TYPES = {"0": "UNKNOWN_RPC", "1": "GRPC", "2": "WEBSOCKET", "3": "JSON_RPC", "4": "REST", "5": "COMET_BFT"}


def nat(value, label):
    if isinstance(value, bool) or not str(value).isdigit():
        raise ValueError(f"{label} must be a nonnegative integer, got {value!r}")
    return int(value)


def dt(value):
    # dt and iso repeat pokt_live_collector's, so that collector can import this module.
    d = datetime.fromisoformat(value.replace("Z", "+00:00"))
    return d.replace(tzinfo=timezone.utc) if d.tzinfo is None else d.astimezone(timezone.utc)


def iso(t):
    return t.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


def check_range(result, start, end):
    """Every catalog answer must cover the asked starts, with no settlement gap."""
    coverage = result["range"]
    if (coverage.get("gaps") or not coverage.get("covered_from") or not coverage.get("covered_to")
            or dt(coverage["covered_from"]) > dt(start) or dt(coverage["covered_to"]) < dt(end)):
        raise ValueError("Session catalog does not cover the window or reports a settlement gap")
    return result["data"]


def roster_page(client, start, end, services=None, session_ids=None, before=None, applications=None):
    before = before or (None, None)
    result = client.gql(ROSTERS_QUERY, a=start, b=end, services=services, ids=session_ids, apps=applications,
                        h=None if before[0] is None else str(before[0]), sid=before[1])["getSessionRostersJson"]
    rows = check_range(result, start, end) if session_ids is None else result["data"]
    if not isinstance(rows, list) or len(rows) > PAGE_SIZE:
        raise ValueError("Invalid or unordered session catalog page")
    keys = [(nat(r["start_height"], "start height"), r["session_id"]) for r in rows]
    if any(b >= a for a, b in zip(keys, keys[1:])) or (before[0] is not None and keys and keys[0] >= (before[0], before[1])):
        raise ValueError("Invalid or unordered session catalog page")
    return rows


def session_claims(row, config, service):
    """One settled claim per member in a settled state, as the indexed claim records were."""
    if row["service_id"] != service or not all(isinstance(row.get(k), str) and row[k] for k in ("session_id", "application_id")):
        raise ValueError("Session identity is missing or belongs to another service")
    end, size = nat(row["end_height"], "session end"), nat(row["roster_size"], "roster size")
    members = list(zip(row["member_ids"], row["member_states"], row["member_paid_upokt"], row["member_owner_ids"]))
    if size != len(members) or len(set(row["member_ids"])) != size or not size:
        raise ValueError("Empty, duplicate or inconsistent catalog roster")
    if row["money_missing"] or "unresolved" in row["member_states"]:
        raise ValueError("Session money is not indexed yet; keep this run for investigation")
    claims = [m for m in members if m[1] in SETTLED_STATES]
    if not claims or row["phase"] != "settled":
        return []
    block = nat(row["settled_height"], "settlement block")
    if not end < block <= config["cutoff"]:
        return []
    return [{"id": f"{row['session_id']}:{supplier}", "supplierId": supplier, "serviceId": service,
             "applicationId": row["application_id"], "sessionId": row["session_id"], "sessionEndHeight": str(end),
             "blockId": str(block), "payoutUpokt": str(nat(paid, "payout")), "payoutVerified": True}
            for supplier, state, paid, _ in claims]


def settled_seats(rows, start, end):
    """Members with a settled claim in the sessions that started in [start, end)."""
    return sum(state in SETTLED_STATES for r in rows if dt(start) <= dt(r["start_time"]) < dt(end)
               for state in r["member_states"])


def network_settled_seats(client, start, end, service):
    """The catalog's own total of settled seats for one service's sessions started in [start, end)."""
    result = client.gql(NETWORK_QUERY, a=start, b=end, services=[service])["getNetworkSessionsJson"]
    rows = check_range(result, start, end)
    if any(r["service_id"] != service or nat(r["active"], "active") or nat(r["unresolved"], "unresolved") for r in rows):
        raise ValueError("Network session total belongs to another service or is not settled")
    return sum(nat(r["paid"], "paid seats")+nat(r["settled_zero"], "settled-zero seats") for r in rows)


def is_statement_timeout(exc):
    message = str(exc)
    return (message.startswith("GraphQL errors:") and "57014" in message
            and "canceling statement due to statement timeout" in message)


def param_at(history, namespace, key, height):
    rows = [r for r in history if r["namespace"] == namespace and r["key"] == key and nat(r["activeAt"], "param height") <= height]
    if not rows:
        raise ValueError(f"No {namespace}.{key} in effect at {height}")
    return nat(max(rows, key=lambda r: nat(r["activeAt"], "param height"))["value"], key)


def supplier_configs(client, members, progress=print):
    """Service config history of each (supplier, service) pair, 200 suppliers per call.

    One service per call: eight services at once can exceed the indexer's statement timeout.
    """
    configs = defaultdict(list)
    by_service = defaultdict(set)
    for supplier, service in members:
        by_service[service].add(supplier)
    def fetch(ids, service, progress):
        try:
            return client.gql(CONFIG_QUERY, ids=ids, services=[service])["getSuppliersConfigHistoryJson"]
        except ValueError as exc:
            if not is_statement_timeout(exc) or len(ids) < 2:
                raise
            # Only the observed PostgreSQL timeout is retried, once, as two halves.
            progress(f"Indexer statement timeout: config history of {len(ids)} {service} suppliers retried as two halves.")
            half = len(ids)//2
            return [row for part in (ids[:half], ids[half:])
                    for row in client.gql(CONFIG_QUERY, ids=part, services=[service])["getSuppliersConfigHistoryJson"]]
    for service, suppliers in sorted(by_service.items()):
        ids = sorted(suppliers)
        for offset in range(0, len(ids), 200):
            for row in fetch(ids[offset:offset+200], service, progress):
                if row["service_id"] != service or row["supplier_id"] not in suppliers:
                    raise ValueError("Config history returned another supplier or service")
                configs[(row["supplier_id"], service)].append(row)
    return configs


def config_at(configs, supplier, service, start):
    """Endpoints the session was served from. A config declared at height d takes
    effect at the next session start, so the one declared at start-1 applies."""
    rows = [r for r in configs.get((supplier, service), []) if nat(r["first_height"], "config start") < start
            and (r["last_height"] is None or start <= nat(r["last_height"], "config end")+1)]
    if len(rows) != 1:
        raise ValueError("A selected Supplier lacks its service-specific historical configuration")
    return [{"url": e["url"], "rpc_type": RPC_TYPES[str(e["rpcType"])]} for e in rows[0]["endpoints"]]


def staked_suppliers(client, height, statuses=None):
    """Suppliers in the given stake statuses at one height, with their owner and service ids."""
    rows, after = [], None
    for _ in range(1000):
        page = client.gql(STAKED_QUERY, h=str(height), after=after, statuses=statuses)["getStakedSuppliersList"]
        if any(int(r["height"]) != height for r in page) or (rows and page and page[0]["supplierId"] <= rows[-1]["supplierId"]):
            raise ValueError("Staked supplier page read another height or is unordered")
        rows.extend(page)
        if len(page) < 1000:
            return rows
        after = page[-1]["supplierId"]
    raise ValueError("Staked supplier pagination failed to finish")


def block_times(client, heights, config, window=True):
    times={}
    values=sorted({int(h) for h in heights})
    for offset in range(0,len(values),200):
        part=values[offset:offset+200]
        conn=client.gql(BLOCKS_QUERY,filter={"id":{"in":[str(h) for h in part]}})["blocks"]
        rows=conn["nodes"]
        if nat(conn["totalCount"],"block count")!=len(part) or len(rows)!=len(part) or conn["pageInfo"]["hasNextPage"] is not False:
            raise ValueError("Historical block timestamp batch is incomplete")
        for row in rows:
            height=nat(row["id"],"block height")
            at=iso(dt(row["timestamp"]))
            if height not in part or height in times or (window and not dt(config["start"])<=dt(at)<dt(config["endExclusive"])):
                raise ValueError("Historical block timestamp is repeated or out of scope")
            times[height]=at
    return times

```

### collector/settlement_collect.py

```python
#!/usr/bin/env python3
"""Claim-first, read-only extraction; does not replace the sampled website.

Python 3.10+, standard library only. Uses the original collector's audited HTTP
cache, retries and endpoint parser. Every session of the pocketdex session
catalog in the service/session-end window is paginated, regardless of
Application; members in a settled state are its settled claims. Historical
membership comes from each session's catalog roster and the members' service
config history, not an interpolated Supplier registry. Amounts remain integer uPOKT.
"""
from __future__ import annotations

import argparse
import gzip
import hashlib
import json
import math
import sqlite3
import time
from collections import Counter, defaultdict
from datetime import timedelta
from pathlib import Path

from pokt_live_collector import (
    Client, FIXED_SERVICES, GRAPHQL,
    atomic_json, digest, dt, iso, normalized_registry, read_json, suffix_rules, utcnow,
)
from session_catalog import (
    PAGE_SIZE, PARAMS_QUERY, ROSTER_ERA_HEIGHT, block_times, config_at, nat, network_settled_seats, param_at,
    roster_page, session_claims, settled_seats, supplier_configs,
)

VERSION = "settlement-first-1.0"
PRODUCT_SHA256 = "feeebfbb120044a782dd60f1879c0613924cb259b7eb528d5f74659055842088"
CUTOFF = 955836
MONTH_START = "2026-09-06T00:00:00Z"
END = "2026-10-06T00:00:00Z"
PILOT_START = "2026-09-29T00:00:00Z"


def log(message):
    print(message,flush=True)


def plan(stage="pilot"):
    if stage not in {"pilot", "month"}:
        raise ValueError("Choose pilot or month")
    return {"version": VERSION, "stage": stage, "cutoff": CUTOFF,
            "start": PILOT_START if stage == "pilot" else MONTH_START,
            "endExclusive": END, "services": ["eth"] if stage == "pilot" else FIXED_SERVICES,
            "membershipTime": "session catalog roster; owner and endpoints at session start",
            "scope": "all indexed settled claims in service/session-end window",
            "referenceProductSha256": PRODUCT_SHA256}


def resolve_boundaries(client, config):
    fields = []
    for alias, value in (("start", config["start"]), ("end", config["endExclusive"])):
        upper = iso(dt(value) + timedelta(minutes=10))
        fields.append(alias+':blocks(first:100,filter:{timestamp:{greaterThanOrEqualTo:'+json.dumps(value)+',lessThan:'+json.dumps(upper)+'}})'
                      '{totalCount pageInfo{hasNextPage} nodes{id timestamp}}')
    data = client.gql("{" + " ".join(fields) + "}")
    bounds = {}
    for alias, value in (("start", config["start"]), ("end", config["endExclusive"])):
        conn = data[alias]
        rows = sorted(conn["nodes"], key=lambda r: nat(r["id"], "block height"))
        if not rows or nat(conn["totalCount"], "boundary count") != len(rows) or conn["pageInfo"]["hasNextPage"] is not False:
            raise ValueError("UTC boundary is empty or truncated")
        if any(not dt(value) <= dt(r["timestamp"]) < dt(value)+timedelta(minutes=10) for r in rows):
            raise ValueError("Boundary timestamp outside its bracket")
        if any(nat(b["id"], "height") != nat(a["id"], "height")+1 or dt(b["timestamp"]) < dt(a["timestamp"])
               for a, b in zip(rows, rows[1:])):
            raise ValueError("Missing, repeated or unordered boundary blocks")
        bounds[alias] = {"height": nat(rows[0]["id"], "height"), "timestamp": iso(dt(rows[0]["timestamp"]))}
    if not bounds["start"]["height"] < bounds["end"]["height"] <= config["cutoff"]:
        raise ValueError("Boundary heights disagree with the fixed cutoff")
    return bounds


def connect(path, config):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    db = sqlite3.connect(path)
    db.execute("PRAGMA foreign_keys=ON")
    db.executescript('''
    CREATE TABLE IF NOT EXISTS config (id INTEGER PRIMARY KEY CHECK(id=1), json TEXT NOT NULL);
    CREATE TABLE IF NOT EXISTS services (service TEXT PRIMARY KEY,total INTEGER,status TEXT NOT NULL);
    CREATE TABLE IF NOT EXISTS pages (service TEXT,page INTEGER,next_cursor TEXT,row_count INTEGER,hash TEXT,
      PRIMARY KEY(service,page));
    CREATE TABLE IF NOT EXISTS claims (id TEXT PRIMARY KEY,session TEXT,service TEXT,app TEXT,
      end_height INTEGER,supplier TEXT,json TEXT NOT NULL,page INTEGER,position INTEGER,UNIQUE(session,supplier));
    CREATE INDEX IF NOT EXISTS claim_session ON claims(session);
    CREATE TABLE IF NOT EXISTS sessions (session TEXT PRIMARY KEY,service TEXT NOT NULL,json TEXT NOT NULL);
    CREATE TABLE IF NOT EXISTS rosters (session TEXT PRIMARY KEY,json TEXT NOT NULL,hash TEXT NOT NULL);
    ''')
    encoded = json.dumps(config, sort_keys=True)
    existing = db.execute("SELECT json FROM config WHERE id=1").fetchone()
    if existing and existing[0] != encoded:
        db.close()
        raise ValueError("Resume database belongs to another plan")
    with db:
        db.execute("INSERT OR IGNORE INTO config VALUES(1,?)", (encoded,))
    return db


def download_service(client, db, config, service, bounds, progress=log):
    # Sessions are selected by END height; the catalog pages by session START, so the
    # start range opens one hour early and the window is applied to end_height here.
    # It also closes two hours late: full coverage there proves the catalog has written
    # every session that ends inside the window.
    start = iso(dt(bounds["start"]["timestamp"])-timedelta(hours=1))
    end = iso(dt(bounds["end"]["timestamp"])+timedelta(hours=2))
    state = db.execute("SELECT total,status FROM services WHERE service=?", (service,)).fetchone()
    if state and state[1] == "complete":
        return state[0]
    previous = db.execute("SELECT page,next_cursor FROM pages WHERE service=? ORDER BY page DESC LIMIT 1", (service,)).fetchone()
    stored = db.execute("SELECT count(*) FROM claims WHERE service=?", (service,)).fetchone()[0]
    with db:
        db.execute("INSERT OR IGNORE INTO services VALUES(?,?,?)", (service, None, "collecting"))
    page = previous[0]+1 if previous else 0
    # The cursor also carries the oldest session end seen and the settled seats counted
    # so far, so a resumed run keeps both checks below.
    before, oldest, seats = None, None, 0
    if previous:
        height, session, oldest, seats = previous[1].split(":")
        before, oldest, seats = (int(height), session), int(oldest), int(seats)
    for _ in range(10000):
        rows = roster_page(client, start, end, [service], before=before)
        kept = [r for r in rows if bounds["start"]["height"] <= nat(r["end_height"], "end") < bounds["end"]["height"]]
        claims = [c for r in kept for c in session_claims(r, config, service)]
        times = block_times(client, [c["blockId"] for c in claims], config, window=False) if claims else {}
        for c in claims:
            c["block"] = {"timestamp": times[int(c["blockId"])]}
        done = len(rows) < PAGE_SIZE
        oldest = nat(rows[-1]["end_height"], "end") if rows else oldest
        seats += settled_seats(rows, config["start"], config["endExclusive"])
        if done and (oldest is None or oldest >= bounds["start"]["height"]):
            raise ValueError("Session catalog start range does not reach before the window")
        # Completeness: the pages hold every settled seat the catalog totals for these session starts.
        if done and seats != network_settled_seats(client, config["start"], config["endExclusive"], service):
            raise ValueError("Session catalog pages and its network total disagree; keep this run for investigation")
        stored += len(claims)
        cursor = f"{rows[-1]['start_height']}:{rows[-1]['session_id']}:{oldest}:{seats}" if rows else None
        # A whole page and its checkpoint commit together. Duplicates roll back.
        with db:
            for r in kept:
                if any(c["sessionId"] == r["session_id"] for c in claims):
                    db.execute("INSERT INTO sessions VALUES(?,?,?)", (r["session_id"], service, json.dumps(r, separators=(",", ":"))))
            for position, c in enumerate(claims):
                db.execute("INSERT INTO claims VALUES(?,?,?,?,?,?,?,?,?)",
                           (c["id"], c["sessionId"], service, c["applicationId"], int(c["sessionEndHeight"]),
                            c["supplierId"], json.dumps(c, separators=(",", ":")), page, position))
            db.execute("INSERT INTO pages VALUES(?,?,?,?,?)", (service, page, cursor, len(claims), digest(claims)))
            if done:
                db.execute("UPDATE services SET total=?,status='complete' WHERE service=?", (stored, service))
        if page % 5 == 0 or done:
            progress(f"{service}: {stored:,} settled claims · catalog page {page+1}; cached progress retained")
        if done:
            return stored
        before, page = (nat(rows[-1]["start_height"], "start"), rows[-1]["session_id"]), page+1
    raise ValueError("Session catalog extraction exceeded the page limit")


def fetch_rosters(client, db, config, progress=log):
    sessions = [json.loads(r[0]) for r in db.execute("SELECT json FROM sessions ORDER BY session")]
    saved = {r[0] for r in db.execute("SELECT session FROM rosters")}
    remaining = [r for r in sessions if r["session_id"] not in saved]
    times = block_times(client, [r["end_height"] for r in remaining], config)
    history = client.gql(PARAMS_QUERY, keys=["num_suppliers_per_session"])["getParamHistoryList"] if remaining else []
    configs = supplier_configs(client, [(m, r["service_id"]) for r in remaining for m in r["member_ids"]], progress)
    with db:
        for r in remaining:
            end, start, service = nat(r["end_height"], "end"), nat(r["start_height"], "start"), r["service_id"]
            if not 1 < start <= end:
                raise ValueError("Historical session header is invalid")
            cap = param_at(history, "session", "num_suppliers_per_session", start)
            if nat(r["roster_size"], "roster size") > cap:
                raise ValueError("Historical roster exceeds the effective session cap")
            # The roster and its configs are fixed at the session start; the owner is the
            # catalog's owner at the start (the archive roster read it at the end height).
            members = normalized_registry([{"operator_address": supplier, "owner_address": owner,
                                            "services": [{"service_id": service, "endpoints": config_at(configs, supplier, service, start)}]}
                                           for supplier, owner in zip(r["member_ids"], r["member_owner_ids"])], [service])
            if {m["id"] for m in members} != set(r["member_ids"]):
                raise ValueError("A selected Supplier lacks its service-specific historical configuration")
            at = times[end]
            if not dt(config["start"]) <= dt(at) < dt(config["endExclusive"]):
                raise ValueError("Session end block does not match the exact UTC window")
            roster = {"id": r["session_id"], "serviceId": service, "applicationId": r["application_id"], "endHeight": end,
                      "startHeight": start, "endedAt": at, "selectedCount": len(members), "selectionCap": cap,
                      "members": members, "membershipSource": "session catalog roster and supplier config history at session start"}
            db.execute("INSERT INTO rosters VALUES(?,?,?)", (roster["id"], json.dumps(roster, separators=(",", ":")), digest(roster)))
    progress(f"Historical rosters: {len(sessions):,}/{len(sessions):,} · owner and endpoint membership verified")
    for sid, text, checksum in db.execute("SELECT session,json,hash FROM rosters"):
        roster = json.loads(text)
        if digest(roster) != checksum:
            raise ValueError("Cached historical roster checksum mismatch")
        ids = {m["id"] for m in roster["members"]}
        if not {r[0] for r in db.execute("SELECT supplier FROM claims WHERE session=?", (sid,))}.issubset(ids):
            raise ValueError("A settled Supplier is absent from the historical roster")
    return len(sessions)


def load_reference(path):
    reference = read_json(path)
    if reference["productSha256"] != PRODUCT_SHA256 or reference["cutoff"] != CUTOFF:
        raise ValueError("Expected the exact original monthly reference")
    return reference


def reconcile_reference(db, config, reference):
    compared, covered, membership_differences = 0, 0, []
    suppliers = reference["supplierIds"]
    for original in reference["sessions"]:
        if original["serviceId"] not in config["services"] or not dt(config["start"]) <= dt(original["observedAt"]) < dt(config["endExclusive"]):
            continue
        # Original sampled sessions at month edges use numeric end membership.
        rows = [json.loads(r[0]) for r in db.execute("SELECT json FROM claims WHERE session=?", (original["id"],))]
        if len(rows) != original["settledClaimCount"]:
            raise ValueError(f"Original indexed claim count changed for {original['id']}")
        compared += 1
        if original["payoutStatus"] != "complete":
            continue
        actual = {r["supplierId"]: nat(r["payoutUpokt"], "payout") for r in rows}
        expected = {suppliers[i]: nat(v, "reference payout") for i, v in original["payouts"]}
        if actual != expected or not all(r["payoutVerified"] for r in rows):
            raise ValueError("Original Supplier distributions do not reconcile exactly in uPOKT")
        saved = db.execute("SELECT json FROM rosters WHERE session=?", (original["id"],)).fetchone()
        if not saved:
            raise ValueError("Overlapping settled session lacks historical roster")
        roster = json.loads(saved[0])
        if {m["id"] for m in roster["members"]} != {suppliers[i] for i, _, _ in original["members"]}:
            raise ValueError("Original selected roster does not reconcile")
        old = {suppliers[i]: (reference["ownerIds"][o], set(endpoints)) for i, o, endpoints in original["members"]}
        for m in roster["members"]:
            current = (m["owner"], {e["id"] for e in m["endpoints"]})
            if current != old[m["id"]]:
                membership_differences.append({"sessionId": original["id"], "supplierId": m["id"],
                    "reason": "sample-time and session-end owner/endpoint observations differ"})
        covered += 1
    return {"sampledSessionsChecked": compared, "coveredSessionsReconciled": covered,
            "membershipDifferences": membership_differences,
            "passed": compared > 0 and covered > 0 and not membership_differences}


def compact_evidence(db, config):
    supplier_index, owner_index, endpoints, sessions = {}, {}, {}, []
    verified_claims, unknown_claims = 0, 0
    group_counts = Counter()
    for (text,) in db.execute("SELECT json FROM rosters ORDER BY session"):
        roster = json.loads(text)
        claims = [json.loads(r[0]) for r in db.execute("SELECT json FROM claims WHERE session=? ORDER BY supplier", (roster["id"],))]
        verified = all(c["payoutVerified"] for c in claims)
        verified_claims += sum(c["payoutVerified"] for c in claims)
        unknown_claims += sum(not c["payoutVerified"] for c in claims)
        memberships = []
        for member in sorted(roster["members"], key=lambda m: m["id"]):
            si = supplier_index.setdefault(member["id"], len(supplier_index))
            oi = owner_index.setdefault(member["owner"], len(owner_index))
            for e in member["endpoints"]:
                if e["id"] in endpoints and endpoints[e["id"]] != e:
                    raise ValueError("Endpoint identity collision")
                endpoints[e["id"]] = e
            memberships.append([si, oi, sorted(e["id"] for e in member["endpoints"])])
        payouts = [[supplier_index[c["supplierId"]], c["payoutUpokt"]] for c in claims] if verified else []
        pool = str(sum(nat(c["payoutUpokt"], "payout") for c in claims)) if verified else None
        for c in claims:
            group_counts[(c["serviceId"], c["applicationId"])] += 1
        sessions.append({"id": roster["id"], "serviceId": roster["serviceId"], "applicationId": roster["applicationId"],
            "sessionEndHeight": roster["endHeight"], "endedAt": roster["endedAt"], "selectedCount": roster["selectedCount"],
            "members": memberships, "payouts": payouts, "poolUpokt": pool,
            "indexedClaimCount": len(claims), "payoutStatus": "verified" if verified else "unknown",
            "outcomeAvailableAt": max(iso(dt(c["block"]["timestamp"])) for c in claims)})
    return {"schemaVersion": 1, "codec": "settlement-session-v1", "dataKind": "live",
        "version": VERSION, "config": config, "supplierIds": list(supplier_index), "ownerIds": list(owner_index),
        "endpointCatalog": sorted(endpoints.values(), key=lambda e: e["id"]),
        "sessions": sorted(sessions, key=lambda s: (s["endedAt"], s["id"])),
        "groups": [{"serviceId": s, "applicationId": a, "claimCount": n} for (s, a), n in sorted(group_counts.items())],
        "coverage": {"indexedClaims": verified_claims+unknown_claims, "verifiedDistributionClaims": verified_claims,
                     "unknownDistributionClaims": unknown_claims, "indexedSettledSessions": len(sessions),
                     "verifiedDistributionSessions": sum(s["payoutStatus"] == "verified" for s in sessions)},
        "notice": "Indexed settled-session earnings only. Sessions absent from the index are outside this denominator, not verified zero. This is not all relays or registered-session presence. No eligible population is inferred; selection models remain in the original sampled snapshot."}


def reconcile_monthly_counts(evidence, reference):
    actual = {(g["serviceId"], g["applicationId"]): g["claimCount"] for g in evidence["groups"]}
    known = {(g["serviceId"], g["applicationId"]): g["monthlyClaimCount"] for g in reference["monthlyGroups"]}
    for key, count in known.items():
        if actual.get(key, 0) != count:
            raise ValueError("Full-month service extraction differs from the completed monthly count check")
    return {"knownGroupsReconciled": len(known), "knownGroupClaimRecords": sum(known.values()),
            "additionalApplicationGroups": [{"serviceId": s, "applicationId": a, "claimCount": n}
                                          for (s, a), n in sorted(actual.items()) if (s, a) not in known]}


def verify_database(db):
    for service, total, status in db.execute("SELECT service,total,status FROM services"):
        if status != "complete" or db.execute("SELECT count(*) FROM claims WHERE service=?", (service,)).fetchone()[0] != total:
            raise ValueError("Incomplete service extraction")
        for page, count, expected in db.execute("SELECT page,row_count,hash FROM pages WHERE service=? ORDER BY page", (service,)):
            rows = [json.loads(r[0]) for r in db.execute("SELECT json FROM claims WHERE service=? AND page=? ORDER BY position", (service, page))]
            if len(rows) != count or digest(rows) != expected:
                raise ValueError("Cached claim page hash or order mismatch")


def verify_pilot_report(path):
    path = Path(path)
    proof = read_json(path)
    if proof.get("status") != "reconciled" or proof.get("config") != plan("pilot") or not proof.get("overlap", {}).get("passed"):
        raise ValueError("A successful matching live pilot report is required")
    if proof.get("coverage", {}).get("unknownDistributionClaims") != 0:
        raise ValueError("Resolve unknown pilot distributions before monthly expansion")
    name = proof.get("artifact", "")
    if not isinstance(name, str) or not name or Path(name).name != name:
        raise ValueError("Pilot evidence must be a local file beside its receipt")
    artifact = path.parent/name
    if not artifact.is_file():
        raise ValueError("Pilot evidence is missing; retain both the receipt and its artifact")
    if hashlib.sha256(artifact.read_bytes()).hexdigest() != proof.get("artifactSha256"):
        raise ValueError("Pilot evidence checksum differs from its receipt")
    evidence = read_json(artifact)
    if (evidence.get("codec") != "settlement-session-v1" or evidence.get("config") != plan("pilot")
            or evidence.get("coverage") != proof["coverage"]):
        raise ValueError("Pilot evidence scope or coverage differs from its receipt")
    return proof


def run(root, stage, reference_path, pilot_report=None, progress=log):
    config = plan(stage)
    reference = load_reference(reference_path)
    if stage == "month":
        if pilot_report is None:
            raise ValueError("Run and reconcile the seven-day live pilot first")
        verify_pilot_report(pilot_report)
    root = Path(root)
    run_dir = root / "runs" / digest(config)[:16]
    run_dir.mkdir(parents=True, exist_ok=True)
    state = {"version": VERSION, "config": config, "status": "running", "startedAt": iso(utcnow())}
    state["collectorSha256"]=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
    started = time.perf_counter()
    client = Client(root, workers=2)
    db = connect(run_dir/"claims.sqlite3", config)
    try:
        suffix_rules()
        meta = client.gql('{_metadata{chain lastProcessedHeight}}', fresh=True)["_metadata"]
        if meta["chain"] != "pocket" or nat(meta["lastProcessedHeight"], "indexer height") < config["cutoff"]:
            raise ValueError("Wrong chain or indexer behind the fixed cutoff")
        bounds = resolve_boundaries(client, config)
        if bounds["start"]["height"] < ROSTER_ERA_HEIGHT:
            raise ValueError("Window predates the catalog's session rosters")
        state.update(boundaryBlocks=bounds, indexerMetadata=meta,
                     rosterSource={"api": GRAPHQL, "function": "getSessionRostersJson", "rostersFromHeight": ROSTER_ERA_HEIGHT})
        atomic_json(run_dir/"run.json", state)
        progress(f"{stage}: {config['start']} to {config['endExclusive']} exclusive · cutoff {config['cutoff']} · {config['services']}")
        for service in config["services"]:
            download_service(client, db, config, service, bounds, progress)
        verify_database(db)
        fetch_rosters(client, db, config, progress)
        overlap = reconcile_reference(db, config, reference)
        evidence = compact_evidence(db, config)
        state.update(overlap=overlap,coverage=evidence["coverage"])
        if not overlap["passed"] or evidence["coverage"]["unknownDistributionClaims"]:
            raise ValueError("Reconciliation needs review; inspect cached claims and membership before expansion")
        monthly = reconcile_monthly_counts(evidence, reference) if stage == "month" else None
        artifact = run_dir/"settlement-evidence.json.gz"
        atomic_json(artifact, evidence)
        state.update(status="reconciled", completedAt=iso(utcnow()), overlap=overlap, monthlyCounts=monthly,
                     coverage=evidence["coverage"], artifact=artifact.name,
                     artifactSha256=hashlib.sha256(artifact.read_bytes()).hexdigest(),
                     metrics={"seconds": round(time.perf_counter()-started, 2), "httpAttempts": len(client.metrics),
                              "wireBytes": sum(m["bytes"] for m in client.metrics), "compressedBytes": artifact.stat().st_size},
                     productUpdate="Not installed. Review the new settled-session scope separately from sampled selection.")
        atomic_json(run_dir/"result.json", state)
        atomic_json(run_dir/"run.json", state)
        progress(json.dumps(state, indent=2))
        return state
    except BaseException as exc:
        state.update(status="failed", errorType=type(exc).__name__, error=str(exc), failedAt=iso(utcnow()))
        atomic_json(run_dir/"run.json", state)
        progress(f"Stopped: {type(exc).__name__}: {exc}. Cached pages and rosters remain in {run_dir}. Rerun the same stage to resume.")
        raise
    finally:
        db.close()


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--stage", choices=["pilot", "month"], default="pilot")
    p.add_argument("--root", type=Path, default=Path("pokt-settlement-work/cache"))
    p.add_argument("--reference", type=Path, required=True)
    p.add_argument("--pilot-report", type=Path)
    p.add_argument("--dry-run", action="store_true")
    args = p.parse_args()
    if args.dry_run:
        print(json.dumps(plan(args.stage), indent=2))
    else:
        run(args.root, args.stage, args.reference, args.pilot_report)


if __name__ == "__main__":
    main()

```

### collector/staged_collection.py

```python
"""Staged notebook orchestration around the unchanged published collector.

7d: the proved Ethereum week plus a bounded paired indexer/on-chain check.
14d: eight services over the final fortnight, after the seven-day checks.
30d: the original fixed monthly plan, after the fourteen-day reconciliation.
All earnings here are conditional on indexed settled sessions. A negative
three-session probe never becomes a verified zero for a whole Application.
"""
from __future__ import annotations

import base64
import binascii
import contextlib
import hashlib
import json
import os
import re
import sqlite3
import threading
import time
import urllib.parse
from datetime import timedelta
from pathlib import Path

import settlement_collect as original
from pokt_live_collector import Client, PAYOUT_REASONS, RPC
from pokt_live_collector import atomic_json, digest, dt, iso, read_json, suffix_rules, utcnow

VERSION = 'staged-settlement-1.0'
SOURCE_COMMIT = 'fc0eb630771b96e157caa38cd687f867dc152dd2'
EXPECTED_COLLECTOR_SHA = '83a42d8abc678dc8893c112f3f4dde37cf260f2e691196ef5e22c940700ccd70'
MODES = ('offline', 'investigate', '7d', '14d', '30d')
RELAY_REASONS = {'TLM_RELAY_BURN_EQUALS_MINT_SUPPLIER_SHAREHOLDER_REWARD_DISTRIBUTION', PAYOUT_REASONS[0]}
GLOBAL_REASONS = {'TLM_GLOBAL_MINT_SUPPLIER_SHAREHOLDER_REWARD_DISTRIBUTION', PAYOUT_REASONS[1]}
UNKNOWN_REASONS = {'', 'UNSPECIFIED', 'UNRECOGNIZED'}


def plan(mode):
    if mode == '7d':
        return original.plan('pilot')
    if mode == '30d':
        return original.plan('month')
    if mode == '14d':
        return {**original.plan('month'), 'stage': 'scale14', 'start': '2026-09-22T00:00:00Z'}
    raise ValueError('Choose 7d, 14d or 30d')


def run_dir(root, mode):
    return Path(root)/'runs'/digest(plan(mode))[:16]


def verify_receipt(path, mode):
    path = Path(path)
    if mode == '7d':
        return original.verify_pilot_report(path)
    proof = read_json(path)
    if proof.get('status') != 'reconciled' or proof.get('config') != plan(mode) or not proof.get('overlap', {}).get('passed'):
        raise ValueError(f'A matching successfully reconciled {mode} receipt is required')
    name = proof.get('artifact')
    if not isinstance(name, str) or not name or Path(name).name != name:
        raise ValueError('Evidence must be a local file beside its receipt')
    artifact = path.parent/name
    if not artifact.is_file() or hashlib.sha256(artifact.read_bytes()).hexdigest() != proof.get('artifactSha256'):
        raise ValueError('Keep the matching receipt and evidence together; evidence checksum differs or is missing')
    evidence = read_json(artifact)
    coverage = evidence.get('coverage', {})
    if evidence.get('codec') != 'settlement-session-v1' or evidence.get('config') != plan(mode) or coverage != proof.get('coverage'):
        raise ValueError('Evidence scope or coverage differs from its receipt')
    if coverage.get('unknownDistributionClaims') != 0 or coverage.get('indexedClaims', 0) <= 0:
        raise ValueError('Resolve unknown distributions before advancing')
    return proof


def choose_pairs(reference, pair_count=3):
    if type(pair_count) is not int or not 1 <= pair_count <= 3:
        raise ValueError('Use one to three session pairs')
    groups = [g for g in reference['monthlyGroups'] if g['serviceId'] == 'eth']
    covered = max(groups, key=lambda g: g['monthlyClaimCount'])
    missing = next(g for g in sorted(groups, key=lambda g: g['applicationId']) if g['monthlyClaimCount'] == 0)
    config = plan('7d')
    rows = [s for s in reference['sessions'] if s['serviceId'] == 'eth'
            and dt(config['start']) <= dt(s['observedAt']) < dt(config['endExclusive'])]
    known = {s['sessionEndHeight']: s for s in rows if s['applicationId'] == covered['applicationId']
             and s['payoutStatus'] == 'complete' and s['settledClaimCount'] > 0}
    unknown = {s['sessionEndHeight']: s for s in rows if s['applicationId'] == missing['applicationId']
               and s['payoutStatus'] == 'missing' and s['settledClaimCount'] == 0}
    ends = sorted(known.keys() & unknown.keys())
    if len(ends) < pair_count:
        raise ValueError('Insufficient matching historical sessions for the paired investigation')
    positions = [0] if pair_count == 1 else [0, len(ends)-1] if pair_count == 2 else [0, len(ends)//2, len(ends)-1]
    return {'covered': covered, 'uncovered': missing,
            'pairs': [{'covered': known[ends[i]], 'uncovered': unknown[ends[i]]} for i in positions]}


def coin_upokt(value):
    if not isinstance(value, str) or not re.fullmatch(r'[0-9]+upokt', value):
        raise ValueError('Expected an exact nonnegative integer upokt coin')
    return int(value[:-5])


def decode_attribute(key, value):
    if not isinstance(key, str) or not isinstance(value, str):
        raise ValueError('Malformed ABCI attribute')
    if not re.fullmatch(r'[a-z_][a-z0-9_]*', key):
        try:
            decoded = base64.b64decode(key, validate=True).decode('utf-8')
            if not re.fullmatch(r'[a-z_][a-z0-9_]*', decoded):
                raise ValueError('Invalid decoded attribute name')
            key = decoded
            value = base64.b64decode(value, validate=True).decode('utf-8')
        except (ValueError, UnicodeError, binascii.Error) as exc:
            raise ValueError('Unsupported ABCI attribute encoding') from exc
    try:
        value = json.loads(value)
    except json.JSONDecodeError:
        pass
    return key, value


def chain_event(event, block_height):
    attrs = {}
    for attribute in event.get('attributes', []):
        key, value = decode_attribute(attribute['key'], attribute['value'])
        if key in attrs:
            raise ValueError('Duplicate settlement-event attribute')
        attrs[key] = value
    identity = ('session_id', 'service_id', 'application_address', 'supplier_operator_address')
    if any(not isinstance(attrs.get(k), str) or not attrs[k] for k in identity):
        raise ValueError('A settlement event lacks an identity field')
    end = original.nat(attrs['session_end_block_height'], 'on-chain session end')
    if end >= block_height:
        raise ValueError('Settlement event precedes the session end')
    detailed = attrs.get('reward_distribution_detailed')
    if not isinstance(detailed, list):
        raise ValueError('Detailed on-chain distribution reasons are unavailable')
    payout = 0
    verified = True
    for transfer in detailed:
        reason = transfer.get('op_reason')
        amount = coin_upokt(transfer.get('amount'))
        if not isinstance(transfer.get('recipient_address'), str) or not transfer['recipient_address']:
            raise ValueError('On-chain distribution recipient is missing')
        if reason in UNKNOWN_REASONS or not isinstance(reason, str):
            verified = False
        if reason in RELAY_REASONS | GLOBAL_REASONS:
            payout += amount
    return {'sessionId': attrs['session_id'], 'serviceId': attrs['service_id'],
            'applicationId': attrs['application_address'], 'supplierId': attrs['supplier_operator_address'],
            'sessionEndHeight': end, 'blockId': block_height, 'payoutUpokt': str(payout),
            'payoutVerified': verified}


def parse_block_results(response, height):
    if response.get('error'):
        raise ValueError('RPC block_results error: '+str(response['error'])[:600])
    result = response.get('result')
    if not isinstance(result, dict) or original.nat(result.get('height'), 'RPC response height') != height:
        raise ValueError('RPC returned an unexpected historical block height')
    buckets = ('finalize_block_events', 'begin_block_events', 'end_block_events')
    if not any(k in result for k in buckets):
        raise ValueError('RPC response has no supported block-event envelope')
    events = []
    for key in buckets:
        value = result.get(key) or []
        if not isinstance(value, list):
            raise ValueError('Malformed RPC event bucket')
        events.extend(value)
    for tx in result.get('txs_results') or []:
        if original.nat(tx.get('code'), 'transaction result code') == 0:
            events.extend(tx.get('events') or [])
    settlements, seen = [], set()
    for event in events:
        if event.get('type') != 'pocket.tokenomics.EventClaimSettled':
            continue
        row = chain_event(event, height)
        key = (row['sessionId'], row['supplierId'])
        if key in seen:
            raise ValueError('Duplicate on-chain settlement for one Supplier/session')
        seen.add(key)
        settlements.append(row)
    return settlements


def claim_key(row):
    return (row['sessionId'], row['supplierId'], row['serviceId'], row['applicationId'],
            int(row['sessionEndHeight']), int(row['blockId']))


def reconcile_control(indexed, on_chain):
    chain = {claim_key(r): r for r in on_chain}
    if len(chain) != len(on_chain):
        raise ValueError('Repeated on-chain claim identity')
    missing, amounts = [], []
    for row in indexed:
        actual = chain.get(claim_key(row))
        if actual is None:
            missing.append(row['supplierId'])
        elif not actual['payoutVerified'] or not row['payoutVerified'] or actual['payoutUpokt'] != row['payoutUpokt']:
            amounts.append(row['supplierId'])
    return {'passed': bool(indexed) and not missing and not amounts,
            'indexedClaimsChecked': len(indexed), 'absentOnChain': missing, 'distributionDifferences': amounts}


def session_claims(client, session, config, bounds):
    rows = original.roster_page(client, None, None, session_ids=[session['id']])
    if len(rows) != 1 or rows[0]['application_id'] != session['applicationId'] or rows[0]['service_id'] != 'eth':
        raise ValueError('Diagnostic catalog row is missing or belongs to another session')
    if original.nat(rows[0]['end_height'], 'session end') != session['sessionEndHeight']:
        raise ValueError('Diagnostic claims returned another historical session')
    return original.session_claims(rows[0], config, 'eth')


def group_claims(client, config, bounds, application, cutoff):
    """Settled claims of one Application's eth sessions ending in the window, as of a cutoff."""
    start = iso(dt(bounds['start']['timestamp'])-timedelta(hours=1))
    end = iso(dt(bounds['end']['timestamp'])+timedelta(hours=2))
    total, before = 0, None
    for _ in range(1000):
        rows = original.roster_page(client, start, end, ['eth'], before=before, applications=[application])
        total += sum(len(original.session_claims(r, {'cutoff': cutoff}, 'eth')) for r in rows
                     if r['application_id'] == application and bounds['start']['height'] <= original.nat(r['end_height'], 'end') < bounds['end']['height'])
        if len(rows) < original.PAGE_SIZE:
            return total
        before = (original.nat(rows[-1]['start_height'], 'start'), rows[-1]['session_id'])
    raise ValueError('Diagnostic group count exceeded its page budget')


def diagnose_missingness(groups, checks, extra_events, errors):
    source_changes = [g['kind'] for g in groups if g['kind'] == 'uncovered' and g['originalCutoffClaims'] > 0]
    failed_controls = any(not c['control']['passed'] for c in checks) or not checks
    discrepancy = bool(extra_events) or bool(source_changes)
    technical = 'passed' if not errors and not failed_controls and not discrepancy else 'needs_review'
    unknown = next((g for g in groups if g['kind'] == 'uncovered'), None)
    if extra_events:
        finding = 'On-chain settlements are absent from the matching indexed session records; investigate the indexer gap.'
    elif source_changes:
        finding = 'The original-cutoff index has changed since the saved reference; reconcile that change before expansion.'
    elif unknown and unknown['laterCutoffClaims'] > unknown['originalCutoffClaims']:
        finding = 'This group has indexed settlements after the original cutoff. The fixed original view still excludes them.'
    else:
        finding = 'No matching uncovered-group settlement was found in the checked blocks. The cause across the full group remains unresolved.'
    return {'technicalStatus': technical, 'safeToCollectIndexedSettlements': technical == 'passed',
            'missingPaymentFinding': finding,
            'positiveCauseEvidenceFound': bool(extra_events) or bool(source_changes) or bool(unknown and unknown['laterCutoffClaims'] > unknown['originalCutoffClaims']),
            'wholeGroupCauseEstablished': False,
            'scope': 'paired historical Ethereum sessions; all group counts are restricted to the same seven-day window',
            'scopeNotice': 'A bounded negative probe is not proof of no work, no actual income or no settlements across an entire group. Never convert the original unknown sampled outcomes into zero.'}


def investigate(root, reference_path, pair_count=3, progress=print, client=None):
    root = Path(root)
    reference = original.load_reference(reference_path)
    selected = choose_pairs(reference, pair_count)
    config = plan('7d')
    target = root/'investigation'
    target.mkdir(parents=True, exist_ok=True)
    client = client or Client(root, workers=2)
    started = time.perf_counter()
    report = {'version': VERSION, 'sourceCommit': SOURCE_COMMIT, 'referenceProductSha256': original.PRODUCT_SHA256,
              'config': config, 'selectedGroups': {k: selected[k] for k in ('covered', 'uncovered')},
              'pairCount': pair_count, 'groups': [], 'checks': [], 'unindexedOnChainEvents': [], 'errors': [],
              'startedAt': iso(utcnow()), 'laterCutoffIsSeparate': True, 'originalSampledCoverageUnchanged': True}
    try:
        metadata = client.gql('{_metadata{chain lastProcessedHeight}}', fresh=True)['_metadata']
        status = client.request(RPC+'/status', label='rpc-status', fresh=True)
        if status.get('error') or status.get('result', {}).get('node_info', {}).get('network') != 'pocket':
            raise ValueError('RPC is unavailable or not Pocket mainnet')
        if metadata['chain'] != 'pocket':
            raise ValueError('Indexer is not Pocket mainnet')
        head = original.nat(status['result']['sync_info']['latest_block_height'], 'RPC head')
        later = min(original.nat(metadata['lastProcessedHeight'], 'indexer height'), head)
        if later < original.CUTOFF:
            raise ValueError('Sources are behind the fixed historical cutoff')
        report.update(indexerMetadata=metadata, laterCutoff=later)
        bounds = original.resolve_boundaries(client, config)
        report['boundaryBlocks'] = bounds
        for kind in ('covered', 'uncovered'):
            group = selected[kind]
            counts = []
            for cutoff in (original.CUTOFF, later):
                counts.append(group_claims(client, config, bounds, group['applicationId'], cutoff))
            report['groups'].append({'kind': kind, 'serviceId': 'eth', 'applicationId': group['applicationId'],
                                     'originalCutoffClaims': counts[0], 'laterCutoffClaims': counts[1]})
        blocks = {}
        for i, pair in enumerate(selected['pairs'], 1):
            progress(f'Investigation: paired historical session {i}/{pair_count}; checking indexer records and actual block events.')
            control = session_claims(client, pair['covered'], config, bounds)
            unknown = session_claims(client, pair['uncovered'], config, bounds)
            if len(control) != pair['covered']['settledClaimCount']:
                raise ValueError('Covered reference claim count changed under the original cutoff')
            # Keep this probe small. Its negative result never certifies a whole group.
            heights = sorted({int(r['blockId']) for r in control + unknown})
            if not heights or len(set(blocks) | set(heights)) > 12:
                raise ValueError('Investigation lacks a positive control or exceeds twelve RPC blocks')
            for height in heights:
                if height not in blocks:
                    response = client.request(RPC+'/block_results?'+urllib.parse.urlencode({'height': height}), label='block-results')
                    blocks[height] = parse_block_results(response, height)
            events = [e for height in heights for e in blocks[height]]
            control_check = reconcile_control(control, events)
            matching = [e for e in events if e['sessionId'] == pair['uncovered']['id']
                        and e['applicationId'] == pair['uncovered']['applicationId'] and e['serviceId'] == 'eth'
                        and e['sessionEndHeight'] == pair['uncovered']['sessionEndHeight']]
            indexed_keys = {claim_key(r) for r in unknown}
            extra = [e for e in matching if claim_key(e) not in indexed_keys]
            report['unindexedOnChainEvents'].extend(extra)
            report['checks'].append({'observedAt': pair['covered']['observedAt'], 'sessionEndHeight': pair['covered']['sessionEndHeight'],
                                    'coveredSessionId': pair['covered']['id'], 'uncoveredSessionId': pair['uncovered']['id'],
                                    'blockHeightsChecked': heights, 'control': control_check,
                                    'uncoveredIndexedClaims': len(unknown), 'uncoveredOnChainClaimsInCheckedBlocks': len(matching)})
            atomic_json(target/'progress.json', report)
        report['rpcBlocksChecked'] = sorted(blocks)
    except Exception as exc:
        report['errors'].append({'type': type(exc).__name__, 'message': str(exc)[:1600]})
    report.update(diagnose_missingness(report['groups'], report['checks'], report['unindexedOnChainEvents'], report['errors']))
    report.update(completedAt=iso(utcnow()), metrics={'seconds': round(time.perf_counter()-started, 2),
                  'httpAttempts': len(client.metrics), 'wireBytes': sum(m['bytes'] for m in client.metrics)})
    atomic_json(target/'result.json', report)
    progress('Investigation technical status: '+report['technicalStatus'])
    progress(report['missingPaymentFinding'])
    return report


def require_investigation(root, reference_path):
    path = Path(root)/'investigation/result.json'
    if not path.is_file():
        raise ValueError('Run MODE="7d" or MODE="investigate" first; the paired investigation is required')
    report = read_json(path)
    groups = choose_pairs(original.load_reference(reference_path))
    expected = {k: groups[k] for k in ('covered', 'uncovered')}
    if (report.get('version') != VERSION or report.get('config') != plan('7d')
            or report.get('referenceProductSha256') != original.PRODUCT_SHA256 or report.get('selectedGroups') != expected
            or report.get('technicalStatus') != 'passed' or report.get('safeToCollectIndexedSettlements') is not True
            or report.get('errors') or report.get('unindexedOnChainEvents') or not report.get('checks')
            or any(not c.get('control', {}).get('passed') for c in report['checks'])):
        raise ValueError('The investigation needs review. Save its result.json and share the exact error before expansion')
    return report


def collect14(root, reference_path, pilot_report, progress=print, client=None):
    original.verify_pilot_report(pilot_report)
    config = plan('14d')
    reference = original.load_reference(reference_path)
    directory = run_dir(root, '14d')
    directory.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    client = client or Client(root, workers=2)
    db = original.connect(directory/'claims.sqlite3', config)
    state = {'version': VERSION, 'config': config, 'status': 'running', 'startedAt': iso(utcnow()),
             'baseCollectorSha256': hashlib.sha256(Path(original.__file__).read_bytes()).hexdigest(), 'sourceCommit': SOURCE_COMMIT}
    try:
        suffix_rules()
        meta = client.gql('{_metadata{chain lastProcessedHeight}}', fresh=True)['_metadata']
        if meta['chain'] != 'pocket' or original.nat(meta['lastProcessedHeight'], 'indexer height') < config['cutoff']:
            raise ValueError('Wrong chain or indexer behind the fixed cutoff')
        bounds = original.resolve_boundaries(client, config)
        if bounds['start']['height'] < original.ROSTER_ERA_HEIGHT:
            raise ValueError("Window predates the catalog's session rosters")
        state.update(indexerMetadata=meta, boundaryBlocks=bounds,
                     rosterSource={'api': original.GRAPHQL, 'function': 'getSessionRostersJson', 'rostersFromHeight': original.ROSTER_ERA_HEIGHT})
        atomic_json(directory/'run.json', state)
        progress('14d: eight services, all indexed Applications; fixed original cutoff.')
        for service in config['services']:
            original.download_service(client, db, config, service, bounds, progress)
        original.verify_database(db)
        original.fetch_rosters(client, db, config, progress)
        overlap = original.reconcile_reference(db, config, reference)
        evidence = original.compact_evidence(db, config)
        if not overlap['passed'] or evidence['coverage']['unknownDistributionClaims']:
            raise ValueError('Reconciliation needs review before monthly expansion')
        artifact = directory/'settlement-evidence.json.gz'
        atomic_json(artifact, evidence)
        state.update(status='reconciled', completedAt=iso(utcnow()), overlap=overlap, coverage=evidence['coverage'],
                     artifact=artifact.name, artifactSha256=hashlib.sha256(artifact.read_bytes()).hexdigest(),
                     metrics={'seconds': round(time.perf_counter()-started, 2), 'httpAttempts': len(client.metrics),
                              'wireBytes': sum(m['bytes'] for m in client.metrics), 'compressedBytes': artifact.stat().st_size},
                     productUpdate='Not installed. Indexed settled-session scope only; original sampled missingness remains unchanged.')
        atomic_json(directory/'result.json', state)
        atomic_json(directory/'run.json', state)
        return state
    except BaseException as exc:
        state.update(status='failed', errorType=type(exc).__name__, error=str(exc)[:1600], failedAt=iso(utcnow()))
        atomic_json(directory/'run.json', state)
        progress('Stopped. Keep this cache and rerun 14d to resume. '+str(exc)[:600])
        raise
    finally:
        db.close()


@contextlib.contextmanager
def single_collector(root, mode):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    path = root/'.staged-collector.lock'
    try:
        with path.open('x', encoding='utf-8') as file:
            json.dump({'pid': os.getpid(), 'mode': mode, 'startedAt': iso(utcnow())}, file)
    except FileExistsError as exc:
        raise RuntimeError('Another staged run owns this cache. Stop it before restarting. If a kernel crash left this lock, confirm no collector is running before removing '+str(path)) from exc
    try:
        yield
    finally:
        path.unlink(missing_ok=True)


def progress_snapshot(root, mode):
    if mode not in ('7d', '14d', '30d'):
        return {'step': 'paired source investigation'}
    database = run_dir(root, mode)/'claims.sqlite3'
    if not database.is_file():
        return {'step': 'source checks / preparation'}
    connection = sqlite3.connect(database.resolve().as_uri()+'?mode=ro', uri=True, timeout=2)
    try:
        return {'downloadedClaims': connection.execute('SELECT count(*) FROM claims').fetchone()[0],
                'historicalRosters': connection.execute('SELECT count(*) FROM rosters').fetchone()[0],
                'services': [{'service': a, 'expectedClaims': b, 'status': c}
                             for a, b, c in connection.execute('SELECT service,total,status FROM services')]}
    finally:
        connection.close()


@contextlib.contextmanager
def heartbeat(root, mode, progress):
    stopped = threading.Event()
    start = time.monotonic()
    def report():
        while not stopped.wait(30):
            try:
                snapshot = progress_snapshot(root, mode)
            except (sqlite3.Error, OSError):
                snapshot = {'step': 'checkpoint temporarily busy'}
            progress(f'Still running ({int(time.monotonic()-start)}s): '+json.dumps(snapshot, separators=(',', ':')))
    worker = threading.Thread(target=report, daemon=True)
    worker.start()
    try:
        yield
    finally:
        stopped.set()
        worker.join(timeout=3)


def execute(mode, root, reference_path, pilot_report, reuse_pilot=True, progress=print):
    if mode not in MODES or mode == 'offline':
        raise ValueError('execute expects investigate, 7d, 14d or 30d')
    if hashlib.sha256(Path(original.__file__).read_bytes()).hexdigest() != EXPECTED_COLLECTOR_SHA:
        raise ValueError('The pinned published settlement collector differs from this notebook')
    with single_collector(root, mode), heartbeat(root, mode, progress):
        if mode == 'investigate':
            return {'mode': mode, 'investigation': investigate(root, reference_path, progress=progress)}
        if mode == '7d':
            if reuse_pilot:
                receipt = original.verify_pilot_report(pilot_report)
                progress('Reusing the included verified Ethereum pilot; no repeat seven-day download.')
            else:
                receipt = original.run(root, 'pilot', reference_path, progress=progress)
                pilot_report = run_dir(root, '7d')/'result.json'
            diagnosis = investigate(root, reference_path, progress=progress)
            return {'mode': mode, 'receipt': receipt, 'receiptPath': str(Path(pilot_report)), 'investigation': diagnosis}
        require_investigation(root, reference_path)
        original.verify_pilot_report(pilot_report)
        if mode == '14d':
            receipt = collect14(root, reference_path, pilot_report, progress)
        else:
            verify_receipt(run_dir(root, '14d')/'result.json', '14d')
            receipt = original.run(root, 'month', reference_path, pilot_report, progress)
        return {'mode': mode, 'receipt': receipt, 'receiptPath': str(run_dir(root, mode)/'result.json')}


def readiness(receipt, investigation=None):
    cov = receipt.get('coverage', {}) if receipt else {}
    complete = bool(receipt and receipt.get('status') == 'reconciled' and receipt.get('overlap', {}).get('passed'))
    return {'collectionReconciled': complete,
            'indexedClaims': cov.get('indexedClaims'), 'indexedSettledSessions': cov.get('indexedSettledSessions'),
            'unknownReturnedDistributionClaims': cov.get('unknownDistributionClaims'),
            'originalSampledSessions': 4080, 'originalComparableSampledSessions': 1652,
            'originalSampledPayoutCoverage': 1652/4080,
            'missingPaymentInvestigation': investigation.get('technicalStatus') if investigation else 'not run',
            'wholeGroupCauseEstablished': investigation.get('wholeGroupCauseEstablished') if investigation else False,
            'fullNetworkEarningsReady': False, 'historicalPredictionAutomaticallyEnabled': False,
            'notice': 'A complete indexed extraction does not resolve absent-index sessions or guarantee enough fleet-specific evidence for a historical earnings test.'}

```

### analysis/settlement_analysis.py

```python
"""Exact integer/Fraction totals for the separate indexed-settlement universe."""
from collections import defaultdict
from fractions import Fraction


def analyze(evidence, dimension="owner", service_id=None):
    if evidence.get("codec")!="settlement-session-v1" or dimension not in {"owner","domain","supplier"}:
        raise ValueError("Use the verified settlement codec and a supported identity dimension")
    suppliers=evidence["supplierIds"];owners=evidence["ownerIds"]
    endpoints={e["id"]:e for e in evidence["endpointCatalog"]}
    sessions=[s for s in evidence["sessions"] if service_id is None or s["serviceId"]==service_id]
    rows={};covered=sum(s["payoutStatus"]=="verified" for s in sessions)
    unassigned=0
    for session in sessions:
        if len(session["members"])!=session["selectedCount"] or len({m[0] for m in session["members"]})!=len(session["members"]):
            raise ValueError("Selected membership does not reconcile with K")
        groups=defaultdict(set)
        for supplier,owner,endpoint_ids in session["members"]:
            if dimension=="owner":labels={owners[owner]}
            elif dimension=="supplier":labels={suppliers[supplier]}
            else:
                labels={endpoints[e]["registrableDomain"] for e in endpoint_ids if endpoints[e]["registrableDomain"]}
                if not labels:unassigned+=1
            for label in labels:groups[label].add(supplier)
        if session["payoutStatus"]=="verified":
            payouts={i:int(v) for i,v in session["payouts"]};pool=int(session["poolUpokt"])
            if sum(payouts.values())!=pool or not set(payouts).issubset({m[0] for m in session["members"]}):
                raise ValueError("Session pool or attributed distributions do not reconcile")
        else:
            if session["poolUpokt"] is not None or session["payouts"]:
                raise ValueError("Unknown payout cannot be filled with an amount")
        for label,selected in groups.items():
            row=rows.setdefault(label,{"label":label,"actualUpokt":0,"reference":Fraction(0),"selectedSupplierIndices":set(),
                "selectedPositions":0,"positiveSessions":0,"informativeDates":set()})
            row["selectedSupplierIndices"].update(selected);row["selectedPositions"]+=len(selected)
            if session["payoutStatus"]=="verified":
                row["actualUpokt"]+=sum(v for i,v in payouts.items() if i in selected)
                row["reference"]+=Fraction(pool*len(selected),session["selectedCount"])
                if pool>0:row["positiveSessions"]+=1;row["informativeDates"].add(session["endedAt"][:10])
    result=[]
    for row in rows.values():
        ref=row.pop("reference");f=float(Fraction(row["actualUpokt"],1)/ref) if ref else None
        row.update(actualUpokt=str(row["actualUpokt"]),actualPokt=int(row["actualUpokt"])/1e6,
            equalSharePokt=float(ref)/1e6,equalShareUpokt={"numerator":str(ref.numerator),"denominator":str(ref.denominator)},
            fScore=f,differencePercent=(f-1)*100 if f is not None else None,
            selectedSupplierIdentities=len(row.pop("selectedSupplierIndices")),informativeDays=len(row.pop("informativeDates")))
        result.append(row)
    return {"dimension":dimension,"serviceId":service_id,"rows":sorted(result,key=lambda r:(-int(r["actualUpokt"]),r["label"])),
        "indexedSettledSessions":len(sessions),"verifiedDistributionSessions":covered,
        "indexedSessionCoverage":covered/len(sessions) if sessions else None,
        "overlapping":dimension=="domain","positionsWithoutRegistrableDomain":unassigned,
        "notice":"Conditional on indexed settled sessions; no registered-session presence or network population is inferred. Domain groups may overlap. Ranks are not assigned by this proof analysis."}

```

### analysis/settlement_product.py

```python
"""Lossless wire packing; does not change a session, membership or amount."""
import argparse
import gzip
import hashlib
import json
from pathlib import Path

V1 = "settlement-session-v1"
V2 = "membership-indices-v2"


def pack(evidence):
    if evidence.get("codec") != V1 or "membershipCatalog" in evidence:
        raise ValueError("Expected canonical settlement-session-v1 evidence")
    endpoint_index = {e["id"]: i for i, e in enumerate(evidence["endpointCatalog"])}
    if len(endpoint_index) != len(evidence["endpointCatalog"]):
        raise ValueError("Duplicate endpoint identity")
    catalog, index, sessions = [], {}, []
    for session in evidence["sessions"]:
        members = []
        for supplier, owner, endpoints in session["members"]:
            if (type(supplier) is not int or type(owner) is not int
                    or not 0 <= supplier < len(evidence["supplierIds"])
                    or not 0 <= owner < len(evidence["ownerIds"])):
                raise ValueError("Invalid membership identity index")
            try:
                key = (supplier, owner, tuple(endpoint_index[e] for e in endpoints))
            except KeyError as exc:
                raise ValueError("Unknown membership endpoint") from exc
            if key not in index:
                index[key] = len(catalog)
                catalog.append([supplier, owner, list(key[2])])
            members.append(index[key])
        sessions.append({**session, "members": members})
    return {**evidence, "codec": V2, "membershipCatalog": catalog, "sessions": sessions}


def unpack(evidence):
    if evidence.get("codec") != V2:
        raise ValueError("Expected membership-indices-v2 evidence")
    endpoints = evidence["endpointCatalog"]
    catalog = []
    for supplier, owner, indices in evidence["membershipCatalog"]:
        if (type(supplier) is not int or type(owner) is not int
                or not 0 <= supplier < len(evidence["supplierIds"])
                or not 0 <= owner < len(evidence["ownerIds"])
                or any(type(i) is not int or not 0 <= i < len(endpoints) for i in indices)):
            raise ValueError("Invalid packed membership identity")
        catalog.append([supplier, owner, [endpoints[i]["id"] for i in indices]])
    sessions = []
    for session in evidence["sessions"]:
        if any(type(i) is not int or not 0 <= i < len(catalog) for i in session["members"]):
            raise ValueError("Invalid packed session membership index")
        sessions.append({**session, "members": [catalog[i] for i in session["members"]]})
    return {**{k: v for k, v in evidence.items() if k != "membershipCatalog"},
            "codec": V1, "sessions": sessions}


def export(evidence, path):
    packed = pack(evidence)
    if unpack(packed) != evidence:
        raise ValueError("Lossless product round trip failed")
    raw = json.dumps(packed, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    blob = gzip.compress(raw, mtime=0)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name+".tmp")
    temp.write_bytes(blob)
    temp.replace(path)
    return {"path": str(path), "bytes": len(blob), "sha256": hashlib.sha256(blob).hexdigest(),
            "codec": V2, "losslessRoundTrip": True,
            "membershipConfigurations": len(packed["membershipCatalog"]),
            "notice": "Lossless packaging only; no website update, eligible-population inference or coverage change."}


if __name__ == "__main__":
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("input", type=Path)
    p.add_argument("output", type=Path)
    args = p.parse_args()
    evidence = json.loads(gzip.decompress(args.input.read_bytes()))
    print(json.dumps(export(evidence, args.output), indent=2))

```

### tests/test_settlement.py

```python
"""Failure, resume and attribution cases for the separate settlement path."""
import copy
import gzip
import json
import sqlite3
import sys
import tempfile
import unittest
from pathlib import Path
from unittest.mock import patch

ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT/"collector"))
import settlement_collect as s
import session_catalog as catalog
sys.path.insert(0,str(ROOT/"analysis"))
from settlement_analysis import analyze
from settlement_product import pack, unpack
from pokt_live_collector import address_parts

REFERENCE=s.load_reference(ROOT/"collector/settlement-reference.json.gz")
ORIGINAL=next(r for r in REFERENCE["sessions"] if r["serviceId"]=="eth" and r["payoutStatus"]=="complete" and r["observedAt"]>="2026-09-29")
SUPPLIER=REFERENCE["supplierIds"][ORIGINAL["payouts"][0][0]]
OWNER=REFERENCE["ownerIds"][ORIGINAL["members"][0][1]]
CONFIG=s.plan("pilot")
BOUNDS={"start":{"height":940000,"timestamp":"2026-09-29T00:00:10Z"},"end":{"height":952159,"timestamp":"2026-10-06T00:00:10Z"}}

RANGE={"requested_from":"2026-09-28T23:00:10Z","requested_to":"2026-10-06T02:00:10Z","covered_from":"2026-09-28T23:00:10Z",
       "covered_to":"2026-10-06T02:00:10Z","gaps":[],"end_inclusive":False}

def row(i=1,amount=27,start=950901,state="paid",started="2026-10-05T03:00:00+00:00"):
    """One getSessionRostersJson row: a one-member roster settled at 951000."""
    return {"session_id":f"{i:064x}","service_id":"eth","application_id":"application","start_height":str(start),"start_time":started,
            "end_height":str(start+19),"phase":"settled","settled_height":"951000","money_missing":False,
            "candidates":"1","roster_size":"1","claimed":"1","paid_upokt":str(amount),"member_ids":[SUPPLIER],
            "member_states":[state],"member_paid_upokt":[str(amount)],"member_owner_ids":[OWNER],"member_domains":[None]}

class Pages:
    """Session catalog pages keyed by the before_* cursor of the page before."""
    def __init__(self,pages,fail=None,coverage=RANGE,network=None):
        self.pages=pages;self.before=[];self.fail=fail;self.coverage=coverage;self.network=network
    def gql(self,query,**v):
        if query==catalog.NETWORK_QUERY:
            # The catalog's own total: every settled seat of the sessions started in the window.
            seats=self.network if self.network is not None else s.settled_seats([r for rows,_ in self.pages for r in rows],v["a"],v["b"])
            return {"getNetworkSessionsJson":{"range":{**self.coverage,"covered_from":v["a"],"covered_to":v["b"]},
                    "data":[{"service_id":"eth","paid":str(seats),"settled_zero":"0","active":"0","unresolved":"0"}]}}
        if query==catalog.BLOCKS_QUERY:
            ids=v["filter"]["id"]["in"]
            return {"blocks":{"totalCount":len(ids),"pageInfo":{"hasNextPage":False},
                    "nodes":[{"id":h,"timestamp":"2026-10-05T05:00:00Z"} for h in ids]}}
        key=None if v["h"] is None else f"{v['h']}:{v['sid']}"
        self.before.append(key)
        i=0 if key is None else next(n+1 for n,(rows,_) in enumerate(self.pages) if rows and f"{rows[-1]['start_height']}:{rows[-1]['session_id']}"==key)
        if self.fail==i:raise RuntimeError("HTTP 429: retry budget exhausted")
        return {"getSessionRostersJson":{"range":self.coverage,"data":copy.deepcopy(self.pages[i][0])}}

def older(rows):
    """Close a last page with a session that ends before the window, as the catalog's one-hour margin does."""
    return rows+[row(10**6,start=939000,started="2026-09-28T23:30:00+00:00")]

class SettlementTests(unittest.TestCase):
    def setUp(self):
        self.tmp=tempfile.TemporaryDirectory(dir=ROOT)
        self.db=s.connect(Path(self.tmp.name)/"claims.sqlite3",CONFIG)
        # Four sessions per catalog page keep the paging cases small.
        self.sizes=[patch.object(s,"PAGE_SIZE",4),patch.object(catalog,"PAGE_SIZE",4)]
        for p in self.sizes:p.start()
    def tearDown(self):
        for p in self.sizes:p.stop()
        self.db.close();self.tmp.cleanup()
    def download(self,client):
        return s.download_service(client,self.db,CONFIG,"eth",BOUNDS,lambda _:None)

    def packing_example(self):
        return {"codec":"settlement-session-v1","supplierIds":["a","b"],"ownerIds":["old","new"],
            "endpointCatalog":[{"id":"e1","registrableDomain":"a.example"},{"id":"e2","registrableDomain":"b.example"}],
            "sessions":[{"id":"first","serviceId":"eth","endedAt":"2026-10-04T00:00:00Z","selectedCount":2,
                "members":[[0,0,["e1","e2"]],[1,1,["e2"]]],"payouts":[[0,str(10**25+27)],[1,"0"]],
                "poolUpokt":str(10**25+27),"payoutStatus":"verified"},
                {"id":"later","serviceId":"eth","endedAt":"2026-10-05T00:00:00Z","selectedCount":2,
                "members":[[0,1,["e2"]],[1,1,["e2"]]],"payouts":[],"poolUpokt":None,"payoutStatus":"unknown"}]}

    def test_lossless_packing_retains_owner_endpoint_changes_and_unknowns(self):
        original=self.packing_example();before=copy.deepcopy(original);packed=pack(original)
        self.assertEqual(original,before);restored=unpack(packed);self.assertEqual(restored,original)
        for dimension in ['owner','domain','supplier']:
            self.assertEqual(analyze(restored,dimension),analyze(original,dimension))
        self.assertEqual(packed['sessions'][1]['poolUpokt'],None)
        self.assertNotEqual(packed['sessions'][0]['members'][0],packed['sessions'][1]['members'][0])

    def test_corrupt_packed_indices_are_rejected(self):
        for value in [-1,1.5,True,1000000]:
            packed=pack(self.packing_example());packed['sessions'][0]['members'][0]=value
            with self.assertRaisesRegex(ValueError,'packed session'):unpack(packed)

    def test_config_history_timeout_is_retried_once_as_two_halves(self):
        calls=[]
        class Busy:
            def gql(self,query,**v):
                calls.append(len(v['ids']))
                if len(calls)==1:
                    raise ValueError('GraphQL errors: canceling statement due to statement timeout; code 57014')
                return {'getSuppliersConfigHistoryJson':[]}
        s.supplier_configs(Busy(),[(f'pokt{i}','eth') for i in range(200)],lambda _:None)
        self.assertEqual(calls,[200,100,100])
        calls.clear()
        class AlwaysBusy(Busy):
            def gql(self,query,**v):
                calls.append(len(v['ids']));raise ValueError('GraphQL errors: canceling statement due to statement timeout; code 57014')
        with self.assertRaisesRegex(ValueError,'timeout'):s.supplier_configs(AlwaysBusy(),[(f'pokt{i}','eth') for i in range(200)],lambda _:None)
        self.assertEqual(calls,[200,100])

    def test_schema_errors_are_not_retried_or_hidden(self):
        calls=[]
        class WrongSchema:
            def gql(self,query,**v):
                calls.append(v);raise ValueError('GraphQL errors: Unknown field')
        with self.assertRaisesRegex(ValueError,'Unknown field'):
            s.supplier_configs(WrongSchema(),[('pokt1','eth')],lambda _:None)
        self.assertEqual(len(calls),1)

    def test_plans_keep_the_reviewed_dates_cutoff_and_service_set(self):
        self.assertEqual(s.plan()["services"],["eth"]);self.assertEqual(s.plan()["start"],"2026-09-29T00:00:00Z")
        self.assertEqual(s.plan("month")["services"],s.FIXED_SERVICES)
        self.assertEqual(s.plan("month")["start"],s.MONTH_START)

    def test_session_scope_is_not_restricted_to_sampled_applications(self):
        p=Pages([(older([row(1)]),None)]);self.download(p)
        self.assertEqual(p.before,[None])
        self.assertIn('services:$services',catalog.ROSTERS_QUERY)

    def test_settled_members_are_claims_and_idle_members_are_not(self):
        r=row();r.update(roster_size="2",member_ids=[SUPPLIER,"pokt1idle"],member_states=["paid","idle_others_worked"],
                        member_paid_upokt=["27","0"],member_owner_ids=[OWNER,OWNER])
        claims=s.session_claims(r,CONFIG,"eth")
        self.assertEqual([(c["supplierId"],c["payoutUpokt"]) for c in claims],[(SUPPLIER,"27")])
        zero=row(state="settled_zero",amount=0);self.assertEqual(s.session_claims(zero,CONFIG,"eth")[0]["payoutUpokt"],"0")
        for state in ("expired","discarded","slashed","idle_no_work"):
            self.assertEqual(s.session_claims(row(state=state),CONFIG,"eth"),[])

    def test_large_integer_amounts_are_exact(self):
        value=10**25+27;r=s.session_claims(row(amount=value),CONFIG,"eth")[0]
        self.assertEqual(int(r["payoutUpokt"]),value)

    def test_unindexed_money_stops_collection(self):
        missing=row();missing["money_missing"]=True
        with self.assertRaisesRegex(ValueError,"not indexed"):s.session_claims(missing,CONFIG,"eth")
        with self.assertRaisesRegex(ValueError,"not indexed"):s.session_claims(row(state="unresolved"),CONFIG,"eth")

    def test_sessions_settled_after_the_cutoff_or_not_yet_are_excluded(self):
        late=row();late["settled_height"]="955837"
        self.assertEqual(s.session_claims(late,CONFIG,"eth"),[])
        active=row();active.update(phase="proving",settled_height=None)
        self.assertEqual(s.session_claims(active,CONFIG,"eth"),[])

    def test_inconsistent_rosters_are_rejected(self):
        for field,value in [("service_id","base"),("roster_size","2"),("member_ids",[SUPPLIER,SUPPLIER])]:
            raw=row();raw[field]=value
            if field=="member_ids":raw.update(roster_size="2",member_states=["paid"]*2,member_paid_upokt=["1"]*2,member_owner_ids=[OWNER]*2)
            with self.assertRaises(ValueError):s.session_claims(raw,CONFIG,"eth")

    def test_window_is_applied_to_session_end_height(self):
        inside,before,after=row(1,start=939990),row(2,start=939970),row(3,start=952140)
        self.download(Pages([([after,inside,before],None)]))
        self.assertEqual([r[0] for r in self.db.execute("SELECT session FROM claims")],[inside["session_id"]])

    def test_pages_must_hold_every_settled_seat_the_catalog_totals(self):
        with self.assertRaisesRegex(ValueError,"network total disagree"):self.download(Pages([(older([row(1)]),None)],network=2))
        self.assertEqual(self.db.execute("SELECT status FROM services").fetchone()[0],"collecting")

    def test_resume_keeps_the_margin_and_seat_count_after_an_empty_last_page(self):
        first=[row(1,start=950901),row(2,start=950881),row(3,start=950861),row(10**6,start=939000,started="2026-09-28T23:30:00+00:00")]
        pages=[(first,1),([],None)]
        with self.assertRaisesRegex(RuntimeError,"429"):self.download(Pages(pages,fail=1))
        retry=Pages(pages);self.assertEqual(self.download(retry),3);self.assertEqual(len(retry.before),1)

    def test_catalog_margin_must_reach_before_the_window(self):
        with self.assertRaisesRegex(ValueError,"does not reach"):self.download(Pages([([row(1)],None)]))

    def test_settlement_gaps_and_short_coverage_are_rejected(self):
        for coverage in ({**RANGE,"gaps":[{"from":"2026-10-01T00:00:00Z","to":"2026-10-01T01:00:00Z"}]},
                         {**RANGE,"covered_to":"2026-10-06T00:00:00Z"},{**RANGE,"covered_from":"2026-09-29T00:00:00Z"}):
            with self.assertRaisesRegex(ValueError,"does not cover"):self.download(Pages([(older([row(1)]),None)],coverage=coverage))

    def test_unordered_pages_are_rejected(self):
        with self.assertRaisesRegex(ValueError,"unordered"):self.download(Pages([(older([row(1,start=950801),row(2,start=950901)]),None)]))

    def test_pages_and_resume_handle_every_page(self):
        first=[row(1,start=950901),row(2,start=950881),row(3,start=950861),row(4,start=950841)]
        p=Pages([(first,1),(older([row(5,start=950821)]),None)])
        self.assertEqual(self.download(p),5);s.verify_database(self.db)
        self.assertEqual(p.before,[None,f"950841:{row(4)['session_id']}"])
        again=Pages([]);self.assertEqual(self.download(again),5);self.assertEqual(again.before,[])

    def test_failure_resumes_from_the_committed_cursor(self):
        pages=[([row(1,start=950901),row(2,start=950881),row(3,start=950861),row(4,start=950841)],1),(older([row(5,start=950821)]),None)]
        with self.assertRaisesRegex(RuntimeError,"429"):self.download(Pages(pages,fail=1))
        self.assertEqual(self.db.execute("SELECT count(*) FROM claims").fetchone()[0],4)
        retry=Pages(pages);self.download(retry);self.assertEqual(retry.before,[f"950841:{row(4)['session_id']}"]);s.verify_database(self.db)

    def test_repeated_sessions_are_rejected_before_anything_is_stored(self):
        with self.assertRaisesRegex(ValueError,"unordered"):self.download(Pages([(older([row(1),row(1)]),None)]))
        pages=[([row(1,start=950901),row(2,start=950881),row(3,start=950861),row(4,start=950841)],1),(older([row(4,start=950841)]),None)]
        with self.assertRaisesRegex(ValueError,"unordered"):self.download(Pages(pages))
        self.assertEqual(self.db.execute("SELECT count(*) FROM claims").fetchone()[0],4)

    def test_duplicate_claims_rollback_the_entire_page(self):
        self.db.execute("INSERT INTO claims VALUES(?,?,?,?,?,?,?,?,?)",(f"{2:064x}:{SUPPLIER}","x","base","a",1,"x","{}",0,0));self.db.commit()
        with self.assertRaises(sqlite3.IntegrityError):self.download(Pages([(older([row(1),row(2,start=950881)]),None)]))
        self.assertEqual(self.db.execute("SELECT count(*) FROM claims WHERE service='eth'").fetchone()[0],0)

    def test_modified_cached_claim_is_detected(self):
        self.download(Pages([(older([row()]),None)]))
        self.db.execute("UPDATE claims SET json=replace(json,'\"27\"','\"28\"')");self.db.commit()
        with self.assertRaisesRegex(ValueError,"hash"):s.verify_database(self.db)

    def test_resume_rejects_a_different_frame(self):
        with self.assertRaisesRegex(ValueError,"another plan"):s.connect(Path(self.tmp.name)/"claims.sqlite3",s.plan("month"))

    def test_reference_wrong_cutoff_is_rejected(self):
        target=Path(self.tmp.name)/"ref.json";r=copy.deepcopy(REFERENCE);r["cutoff"]+=1
        target.write_text(json.dumps(r))
        with self.assertRaisesRegex(ValueError,"exact"):s.load_reference(target)

    def test_month_requires_a_successful_live_pilot(self):
        with self.assertRaisesRegex(ValueError,"seven-day"):s.run(Path(self.tmp.name),"month",ROOT/"collector/settlement-reference.json.gz")

    def test_month_requires_the_pilot_artifact_before_http(self):
        path=Path(self.tmp.name)/"result.json"
        path.write_text(json.dumps({"status":"reconciled","config":s.plan(),"overlap":{"passed":True},
            "coverage":{"unknownDistributionClaims":0},"artifact":"absent.json.gz","artifactSha256":"0"*64}))
        with patch.object(s,"Client",side_effect=AssertionError("HTTP must not start")):
            with self.assertRaisesRegex(ValueError,"evidence is missing"):
                s.run(Path(self.tmp.name),"month",ROOT/"collector/settlement-reference.json.gz",path)

    def test_month_rejects_modified_pilot_bytes_before_http(self):
        path=Path(self.tmp.name)/"result.json"
        (path.parent/"evidence.json.gz").write_bytes(b"modified after reconciliation")
        path.write_text(json.dumps({"status":"reconciled","config":s.plan(),"overlap":{"passed":True},
            "coverage":{"unknownDistributionClaims":0},"artifact":"evidence.json.gz","artifactSha256":"0"*64}))
        with patch.object(s,"Client",side_effect=AssertionError("HTTP must not start")):
            with self.assertRaisesRegex(ValueError,"evidence checksum"):
                s.run(Path(self.tmp.name),"month",ROOT/"collector/settlement-reference.json.gz",path)

    def test_monthly_known_counts_reconcile_and_extra_apps_are_retained(self):
        evidence={"groups":[{**g,"claimCount":g["monthlyClaimCount"]} for g in REFERENCE["monthlyGroups"] if g["monthlyClaimCount"]]}
        evidence["groups"].append({"serviceId":"eth","applicationId":"new-between-observations","claimCount":7})
        result=s.reconcile_monthly_counts(evidence,REFERENCE)
        self.assertEqual(result["knownGroupClaimRecords"],851096);self.assertEqual(len(result["additionalApplicationGroups"]),1)
        evidence["groups"][0]["claimCount"]+=1
        with self.assertRaises(ValueError):s.reconcile_monthly_counts(evidence,REFERENCE)

    def test_utc_boundary_is_selected_by_numeric_height(self):
        class Boundary:
            def gql(self,q):return {"start":{"totalCount":2,"pageInfo":{"hasNextPage":False},"nodes":[{"id":"10","timestamp":"2026-09-29T00:00:20Z"},{"id":"9","timestamp":"2026-09-29T00:00:10Z"}]},"end":{"totalCount":1,"pageInfo":{"hasNextPage":False},"nodes":[{"id":"20","timestamp":"2026-10-06T00:00:10Z"}]}}
        self.assertEqual(s.resolve_boundaries(Boundary(),CONFIG)["start"]["height"],9)

    def recorded(self):
        fixture=json.loads(gzip.decompress((ROOT/'tests/fixtures/settlement-catalog-roster.json.gz').read_bytes()))
        calls=[]
        class Catalog:
            def gql(self,query,**v):
                calls.append((query,v))
                if query==catalog.PARAMS_QUERY:return {'getParamHistoryList':copy.deepcopy(fixture['params'])}
                if query==catalog.BLOCKS_QUERY:
                    ids=v['filter']['id']['in']
                    return {'blocks':{'totalCount':len(ids),'pageInfo':{'hasNextPage':False},'nodes':[{'id':h,'timestamp':fixture['blocks'][h]} for h in ids]}}
                return {'getSuppliersConfigHistoryJson':[r for r in copy.deepcopy(fixture['configs']) if r['supplier_id'] in v['ids'] and r['service_id'] in v['services']]}
        row=fixture['session']
        self.db.execute('INSERT INTO sessions VALUES(?,?,?)',(row['session_id'],'eth',json.dumps(row)));self.db.commit()
        return fixture,Catalog(),calls

    def test_live_recorded_catalog_roster_replays_without_network(self):
        fixture,client,calls=self.recorded()
        self.assertEqual(s.fetch_rosters(client,self.db,CONFIG,lambda _:None),1)
        roster=json.loads(self.db.execute('SELECT json FROM rosters').fetchone()[0])
        self.assertEqual(len(roster['members']),50);self.assertEqual(roster['selectionCap'],50)
        self.assertTrue(all(m['endpoints'] for m in roster['members']))
        self.assertEqual({q[1]['services'][0] for q in calls if q[0]==catalog.CONFIG_QUERY},{'eth'})
        original=next(r for r in REFERENCE['sessions'] if r['id']==roster['id'])
        self.assertEqual({m['id'] for m in roster['members']},{REFERENCE['supplierIds'][i] for i,_,_ in original['members']})

    def test_missing_member_config_stops_collection(self):
        fixture,client,_=self.recorded()
        fixture['configs'].clear()
        with self.assertRaisesRegex(ValueError,'lacks'):s.fetch_rosters(client,self.db,CONFIG,lambda _:None)

    def test_config_declared_before_the_session_start_is_the_one_served(self):
        # Recorded mainnet rows (2026-09): a restake inside session 913601..913620 and a config
        # that ended inside session 913141..913160; the archive roster showed the earlier config.
        restaked=[{'first_height':'901731','last_height':'913615','endpoints':[{'url':'https://igniter-nodefleet.relayminer.shannon-mainnet.eu.nodefleet.net','rpcType':'3'}]},
                  {'first_height':'913616','last_height':None,'endpoints':[{'url':'https://dopokt1.relayminer.shannon-mainnet.eu.nodefleet.net/','rpcType':'3'}]}]
        ended=[{'first_height':'874060','last_height':'913152','endpoints':[{'url':'https://eu-eth.sgtstake.com','rpcType':'3'}]}]
        configs={('a','eth'):restaked,('b','eth'):ended}
        self.assertEqual(s.config_at(configs,'a','eth',913601)[0]['url'],restaked[0]['endpoints'][0]['url'])
        self.assertEqual(s.config_at(configs,'a','eth',913621)[0]['url'],restaked[1]['endpoints'][0]['url'])
        self.assertEqual(s.config_at(configs,'b','eth',913141),[{'url':'https://eu-eth.sgtstake.com','rpc_type':'JSON_RPC'}])
        # A restake in the start block itself belongs to the next session.
        self.assertEqual(s.config_at(configs,'a','eth',913616)[0]['url'],restaked[0]['endpoints'][0]['url'])
        with self.assertRaisesRegex(ValueError,'lacks'):s.config_at(configs,'b','eth',913161)

    def test_compact_evidence_keeps_an_unknown_session_out_of_earnings(self):
        sid=f"{1:064x}"
        claim={'id':f'{sid}:{SUPPLIER}','supplierId':SUPPLIER,'serviceId':'eth','applicationId':'application','sessionId':sid,
               'sessionEndHeight':'950920','blockId':'951000','payoutUpokt':'27','payoutVerified':False,'block':{'timestamp':'2026-10-05T05:00:00Z'}}
        self.db.execute('INSERT INTO claims VALUES(?,?,?,?,?,?,?,?,?)',(claim['id'],sid,'eth','application',950920,SUPPLIER,json.dumps(claim),0,0))
        roster={'id':sid,'serviceId':'eth','applicationId':'application','endHeight':950920,'endedAt':'2026-10-05T03:00:00Z','selectedCount':1,
          'members':[{'id':SUPPLIER,'owner':OWNER,'endpoints':[]}]}
        self.db.execute('INSERT INTO rosters VALUES(?,?,?)',(roster['id'],json.dumps(roster),s.digest(roster)));self.db.commit()
        evidence=s.compact_evidence(self.db,CONFIG)
        self.assertEqual(evidence['sessions'][0]['payoutStatus'],'unknown')
        self.assertIsNone(evidence['sessions'][0]['poolUpokt']);self.assertEqual(evidence['sessions'][0]['payouts'],[])
        self.assertEqual(evidence['coverage']['unknownDistributionClaims'],1)

    def test_exact_original_payout_and_membership_overlap(self):
        original=copy.deepcopy(ORIGINAL);sid=original['id'];owners=REFERENCE['ownerIds'];ids=REFERENCE['supplierIds']
        r=row();r.update(session_id=sid,application_id=original['applicationId'],end_height=str(original['sessionEndHeight']),
                        roster_size=str(len(original['payouts'])),member_ids=[ids[i] for i,_ in original['payouts']],
                        member_states=['paid']*len(original['payouts']),member_paid_upokt=[v for _,v in original['payouts']],
                        member_owner_ids=[OWNER]*len(original['payouts']),settled_height=str(original['sessionEndHeight']+40))
        for pos,c in enumerate(s.session_claims(r,CONFIG,'eth')):
            c['block']={'timestamp':'2026-10-05T05:00:00Z'}
            self.db.execute('INSERT INTO claims VALUES(?,?,?,?,?,?,?,?,?)',(c['id'],sid,'eth',c['applicationId'],int(c['sessionEndHeight']),c['supplierId'],json.dumps(c),0,pos))
        roster={'id':sid,'members':[{'id':ids[i],'owner':owners[o],'endpoints':[{'id':eid} for eid in eps]} for i,o,eps in original['members']]}
        self.db.execute('INSERT INTO rosters VALUES(?,?,?)',(sid,json.dumps(roster),s.digest(roster)));self.db.commit()
        reference={**REFERENCE,'sessions':[original]}
        self.assertTrue(s.reconcile_reference(self.db,CONFIG,reference)['passed'])
        roster['members'][0]['owner']=owners[(original['members'][0][1]+1)%len(owners)]
        self.db.execute('UPDATE rosters SET json=?,hash=? WHERE session=?',(json.dumps(roster),s.digest(roster),sid));self.db.commit()
        result=s.reconcile_reference(self.db,CONFIG,reference)
        self.assertFalse(result['passed']);self.assertEqual(len(result['membershipDifferences']),1)

    def test_block_timestamp_batches_reject_missing_and_out_of_scope_rows(self):
        class Blocks:
            def gql(self,q,**v):
                return {'blocks':{'totalCount':2,'pageInfo':{'hasNextPage':False},'nodes':[{'id':'950920','timestamp':'2026-10-05T03:00:00Z'},{'id':'950940','timestamp':'2026-10-05T03:20:00Z'}]}}
        result=s.block_times(Blocks(),[950920,950940],CONFIG);self.assertEqual(len(result),2)
        with self.assertRaisesRegex(ValueError,'out of scope'):s.block_times(Blocks(),[950920,950940],{**CONFIG,'endExclusive':'2026-10-05T03:10:00Z'})
        self.assertEqual(len(s.block_times(Blocks(),[950920,950940],{**CONFIG,'endExclusive':'2026-10-05T03:10:00Z'},window=False)),2)
        with self.assertRaisesRegex(ValueError,'incomplete'):s.block_times(Blocks(),[950920],CONFIG)

    def test_owner_partitions_use_exact_equal_share_ratios_and_domains_overlap(self):
        evidence={'codec':'settlement-session-v1','supplierIds':['a','b'],'ownerIds':['one','two'],
          'endpointCatalog':[{'id':'e1','registrableDomain':'a.example'},{'id':'e2','registrableDomain':'b.example'}],
          'sessions':[{'id':'s','serviceId':'eth','endedAt':'2026-10-01T01:00:00Z','selectedCount':2,
            'members':[[0,0,['e1','e2']],[1,1,['e2']]],'payouts':[[0,'30'],[1,'70']],'poolUpokt':'100','payoutStatus':'verified'}]}
        result=analyze(evidence,'owner');by={r['label']:r for r in result['rows']}
        self.assertEqual(sum(int(r['actualUpokt']) for r in result['rows']),100)
        self.assertEqual(by['one']['equalShareUpokt'],{'numerator':'50','denominator':'1'})
        self.assertEqual(by['one']['fScore'],.6);self.assertAlmostEqual(by['one']['differencePercent'],-40)
        domains=analyze(evidence,'domain');self.assertTrue(domains['overlapping'])
        self.assertEqual(sum(int(r['actualUpokt']) for r in domains['rows']),130)

    def test_inconsistent_settlement_pool_is_rejected_before_analysis(self):
        e={'codec':'settlement-session-v1','supplierIds':['a'],'ownerIds':['one'],'endpointCatalog':[],
           'sessions':[{'serviceId':'eth','endedAt':'2026-10-01','selectedCount':1,'members':[[0,0,[]]],'payouts':[[0,'7']],'poolUpokt':'9','payoutStatus':'verified'}]}
        with self.assertRaisesRegex(ValueError,'pool'):analyze(e)


if __name__=="__main__":unittest.main()

```

### tests/test_staged.py

```python
"""Failure-oriented checks for staged gates and independent on-chain evidence."""
import base64
import copy
import gzip
import hashlib
import json
import sqlite3
import sys
import tempfile
import unittest
from pathlib import Path
from unittest.mock import patch

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT/'collector'))
import staged_collection as s
import settlement_collect as original

REFERENCE = original.load_reference(ROOT/'collector/settlement-reference.json.gz')
PILOT_PATH = ROOT/'data/settlement-pilot/result.json' if (ROOT/'data/settlement-pilot/result.json').exists() else ROOT/'reviewed-pilot/result.json'
FIXTURE = json.loads(gzip.decompress((ROOT/'tests/fixtures/settlement-chain-events.json.gz').read_bytes()))


class StagedTests(unittest.TestCase):
    def test_seven_and_month_plans_keep_original_cache_identities(self):
        self.assertEqual(s.plan('7d'), original.plan('pilot'))
        self.assertEqual(s.plan('30d'), original.plan('month'))
        self.assertEqual(s.run_dir(Path('cache'), '30d').name, '157cf9d485d5a879')

    def test_fortnight_is_same_cutoff_all_services_and_exactly_fourteen_days(self):
        plan = s.plan('14d')
        self.assertEqual((s.dt(plan['endExclusive'])-s.dt(plan['start'])).days, 14)
        self.assertEqual(plan['services'], original.FIXED_SERVICES)
        self.assertEqual(plan['cutoff'], original.CUTOFF)
        self.assertNotEqual(s.run_dir('cache', '14d'), s.run_dir('cache', '30d'))

    def test_pair_selection_keeps_group_roles_and_shared_session_end(self):
        picked = s.choose_pairs(REFERENCE)
        self.assertGreater(picked['covered']['monthlyClaimCount'], 0)
        self.assertEqual(picked['uncovered']['monthlyClaimCount'], 0)
        self.assertEqual(len(picked['pairs']), 3)
        for pair in picked['pairs']:
            self.assertEqual(pair['covered']['sessionEndHeight'], pair['uncovered']['sessionEndHeight'])
            self.assertGreater(pair['covered']['settledClaimCount'], 0)
            self.assertEqual(pair['uncovered']['settledClaimCount'], 0)

    def test_pair_budget_rejects_zero_bool_and_unbounded_counts(self):
        for value in (0, True, 4, -1, 1.5):
            with self.assertRaises(ValueError):
                s.choose_pairs(REFERENCE, value)

    def test_real_abci_events_filter_supplier_reasons_exactly(self):
        rows = s.parse_block_results(FIXTURE['response'], 955813)
        self.assertEqual(len(rows), 3)
        self.assertEqual(rows[0]['payoutUpokt'], '17919')
        self.assertTrue(rows[0]['payoutVerified'])
        self.assertEqual(rows[0]['sessionEndHeight'], 955780)

    def test_base64_encoded_abci_attributes_restore_same_evidence(self):
        response = copy.deepcopy(FIXTURE['response'])
        for event in response['result']['finalize_block_events']:
            for attr in event['attributes']:
                attr['key'] = base64.b64encode(attr['key'].encode()).decode()
                attr['value'] = base64.b64encode(attr['value'].encode()).decode()
        self.assertEqual(s.parse_block_results(response, 955813), s.parse_block_results(FIXTURE['response'], 955813))

    def test_wrong_block_and_rpc_error_stop_independent_reconciliation(self):
        with self.assertRaisesRegex(ValueError, 'unexpected'):
            s.parse_block_results(FIXTURE['response'], 955814)
        with self.assertRaisesRegex(ValueError, 'RPC block_results error'):
            s.parse_block_results({'error': {'message': 'pruned height'}}, 955813)

    def test_event_envelope_is_required_even_when_no_settlements_found(self):
        with self.assertRaisesRegex(ValueError, 'envelope'):
            s.parse_block_results({'result': {'height': '955813'}}, 955813)

    def test_duplicate_supplier_session_events_are_rejected(self):
        response = copy.deepcopy(FIXTURE['response'])
        response['result']['finalize_block_events'].append(response['result']['finalize_block_events'][0])
        with self.assertRaisesRegex(ValueError, 'Duplicate on-chain'):
            s.parse_block_results(response, 955813)

    def test_duplicate_attributes_and_missing_reason_details_are_rejected(self):
        event = copy.deepcopy(FIXTURE['response']['result']['finalize_block_events'][0])
        event['attributes'].append(event['attributes'][0])
        with self.assertRaisesRegex(ValueError, 'Duplicate settlement-event'):
            s.chain_event(event, 955813)
        event = copy.deepcopy(FIXTURE['response']['result']['finalize_block_events'][0])
        event['attributes'] = [a for a in event['attributes'] if a['key'] != 'reward_distribution_detailed']
        with self.assertRaisesRegex(ValueError, 'Detailed'):
            s.chain_event(event, 955813)

    def test_failed_transaction_events_do_not_establish_a_payment(self):
        event = FIXTURE['response']['result']['finalize_block_events'][0]
        response = {'result': {'height': '955813', 'finalize_block_events': [],
                               'txs_results': [{'code': 5, 'events': [event]}]}}
        self.assertEqual(s.parse_block_results(response, 955813), [])

    def test_unknown_distribution_reason_is_retained_as_unverified(self):
        event = copy.deepcopy(FIXTURE['response']['result']['finalize_block_events'][0])
        for attr in event['attributes']:
            if attr['key'] == 'reward_distribution_detailed':
                transfers = json.loads(attr['value'])
                transfers.append({'op_reason': 'UNSPECIFIED', 'recipient_address': 'unknown', 'amount': '1upokt'})
                attr['value'] = json.dumps(transfers)
        row = s.chain_event(event, 955813)
        self.assertFalse(row['payoutVerified'])
        self.assertFalse(s.reconcile_control([row], [row])['passed'])

    def test_coin_parser_preserves_large_integers_and_rejects_other_denoms(self):
        self.assertEqual(s.coin_upokt('90071992547409931234upokt'), 90071992547409931234)
        for value in ('-1upokt', '0.1upokt', '1pokt', 1, '1upokt '):
            with self.assertRaises(ValueError):
                s.coin_upokt(value)

    def test_control_must_have_exact_on_chain_identity_and_amount(self):
        rows = s.parse_block_results(FIXTURE['response'], 955813)
        self.assertTrue(s.reconcile_control(rows, rows)['passed'])
        altered = copy.deepcopy(rows)
        altered[0]['payoutUpokt'] = str(int(altered[0]['payoutUpokt'])+1)
        self.assertFalse(s.reconcile_control(rows, altered)['passed'])
        self.assertFalse(s.reconcile_control(rows, rows[1:])['passed'])
        self.assertFalse(s.reconcile_control([], rows)['passed'])

    def test_negative_probe_never_certifies_whole_group_or_fills_zero(self):
        groups = [{'kind': 'covered', 'originalCutoffClaims': 100, 'laterCutoffClaims': 100},
                  {'kind': 'uncovered', 'originalCutoffClaims': 0, 'laterCutoffClaims': 0}]
        result = s.diagnose_missingness(groups, [{'control': {'passed': True}}], [], [])
        self.assertEqual(result['technicalStatus'], 'passed')
        self.assertFalse(result['wholeGroupCauseEstablished'])
        self.assertIn('unresolved', result['missingPaymentFinding'])

    def test_positive_indexer_gap_blocks_expansion(self):
        result = s.diagnose_missingness([], [{'control': {'passed': True}}], [{'supplierId': 'missing'}], [])
        self.assertEqual(result['technicalStatus'], 'needs_review')
        self.assertFalse(result['safeToCollectIndexedSettlements'])

    def test_later_cutoff_settlements_do_not_rewrite_original_scope(self):
        result = s.diagnose_missingness([{'kind': 'uncovered', 'originalCutoffClaims': 0, 'laterCutoffClaims': 7}],
                                       [{'control': {'passed': True}}], [], [])
        self.assertIn('after the original cutoff', result['missingPaymentFinding'])
        self.assertFalse(result['wholeGroupCauseEstablished'])

    def test_changed_original_cutoff_and_api_errors_require_review(self):
        result = s.diagnose_missingness([{'kind': 'uncovered', 'originalCutoffClaims': 4, 'laterCutoffClaims': 4}],
                                       [{'control': {'passed': True}}], [], [])
        self.assertFalse(result['safeToCollectIndexedSettlements'])
        result = s.diagnose_missingness([], [], [], [{'message': 'HTTP 403'}])
        self.assertFalse(result['safeToCollectIndexedSettlements'])

    def test_investigation_failure_is_saved_without_claiming_success(self):
        class Blocked:
            metrics = []
            def gql(self, *args, **kwargs):
                raise RuntimeError('HTTP 403: example access failure')
        with tempfile.TemporaryDirectory() as folder:
            result = s.investigate(folder, ROOT/'collector/settlement-reference.json.gz', client=Blocked(), progress=lambda _: None)
            saved = s.read_json(Path(folder)/'investigation/result.json')
            self.assertEqual(result['technicalStatus'], 'needs_review')
            self.assertEqual(result, saved)
            self.assertIn('HTTP 403', saved['errors'][0]['message'])

    def test_complete_paired_probe_detects_a_realistic_unindexed_payment(self):
        picked = copy.deepcopy(s.choose_pairs(REFERENCE, 1))
        control = picked['pairs'][0]['covered']
        missing = picked['pairs'][0]['uncovered']
        control['settledClaimCount'] = 1
        height = control['sessionEndHeight']+33
        base_event = copy.deepcopy(FIXTURE['response']['result']['finalize_block_events'][0])
        supplier = REFERENCE['supplierIds'][control['members'][0][0]]
        def event_for(session):
            event = copy.deepcopy(base_event)
            values = {'session_id': session['id'], 'service_id': 'eth', 'application_address': session['applicationId'],
                      'supplier_operator_address': supplier, 'session_end_block_height': str(session['sessionEndHeight'])}
            for attr in event['attributes']:
                if attr['key'] in values:
                    attr['value'] = json.dumps(values[attr['key']])
            return event
        def roster(session, state, paid):
            return {'session_id': session['id'], 'service_id': 'eth', 'application_id': session['applicationId'],
                    'start_height': str(session['sessionEndHeight']-19), 'end_height': str(session['sessionEndHeight']),
                    'phase': 'settled', 'settled_height': str(height), 'money_missing': False, 'roster_size': '1',
                    'member_ids': [supplier], 'member_states': [state], 'member_paid_upokt': [paid], 'member_owner_ids': [supplier]}
        rows = {control['id']: roster(control, 'paid', '17919'), missing['id']: roster(missing, 'idle_no_work', '0')}
        coverage = {'covered_from': '2026-09-28T00:00:00Z', 'covered_to': '2026-10-07T00:00:00Z', 'gaps': []}
        class Source:
            metrics = []
            def gql(self, query, **variables):
                if '_metadata' in query:
                    return {'_metadata': {'chain': 'pocket', 'lastProcessedHeight': 959000}}
                found = [r for r in rows.values() if r['session_id'] in (variables['ids'] or [])
                         or r['application_id'] in (variables['apps'] or [])]
                return {'getSessionRostersJson': {'range': coverage, 'data': copy.deepcopy(found)}}
            def request(self, url, **kwargs):
                if url.endswith('/status'):
                    return {'result': {'node_info': {'network': 'pocket'}, 'sync_info': {'latest_block_height': '959000'}}}
                return {'result': {'height': str(height), 'finalize_block_events': [event_for(control), event_for(missing)]}}
        with tempfile.TemporaryDirectory() as folder, patch.object(s, 'choose_pairs', return_value=picked), \
                patch.object(original, 'resolve_boundaries', return_value={'start': {'height': 942171, 'timestamp': '2026-09-29T00:00:10Z'},
                                                                            'end': {'height': 952159, 'timestamp': '2026-10-06T00:00:09Z'}}):
            result = s.investigate(folder, ROOT/'collector/settlement-reference.json.gz', pair_count=1, client=Source(), progress=lambda _: None)
            self.assertEqual(result['errors'], [])
            self.assertTrue(result['checks'][0]['control']['passed'])
            self.assertEqual(len(result['unindexedOnChainEvents']), 1)
            self.assertTrue(result['positiveCauseEvidenceFound'])
            self.assertFalse(result['safeToCollectIndexedSettlements'])
            self.assertFalse(result['wholeGroupCauseEstablished'])

    def test_stage14_without_investigation_stops_before_any_http(self):
        with tempfile.TemporaryDirectory() as folder, patch.object(s.Client, 'request', side_effect=AssertionError('HTTP must not start')):
            with self.assertRaisesRegex(ValueError, 'investigation'):
                s.execute('14d', folder, ROOT/'collector/settlement-reference.json.gz', PILOT_PATH)

    def test_month_without_fortnight_proof_stops_before_any_http(self):
        with tempfile.TemporaryDirectory() as folder, patch.object(s, 'require_investigation'), patch.object(s.Client, 'request', side_effect=AssertionError('HTTP must not start')):
            with self.assertRaises(FileNotFoundError):
                s.execute('30d', folder, ROOT/'collector/settlement-reference.json.gz', PILOT_PATH)

    def test_fortnight_artifact_tampering_and_wrong_scope_fail_gate(self):
        with tempfile.TemporaryDirectory() as folder:
            folder = Path(folder)
            evidence = {'codec': 'settlement-session-v1', 'config': s.plan('14d'),
                        'coverage': {'indexedClaims': 2, 'unknownDistributionClaims': 0}}
            artifact = folder/'settlement-evidence.json.gz'
            original.atomic_json(artifact, evidence)
            proof = {'status': 'reconciled', 'config': s.plan('14d'), 'overlap': {'passed': True},
                     'coverage': evidence['coverage'], 'artifact': artifact.name,
                     'artifactSha256': hashlib.sha256(artifact.read_bytes()).hexdigest()}
            original.atomic_json(folder/'result.json', proof)
            self.assertEqual(s.verify_receipt(folder/'result.json', '14d'), proof)
            artifact.write_bytes(artifact.read_bytes()+b'changed')
            with self.assertRaisesRegex(ValueError, 'checksum'):
                s.verify_receipt(folder/'result.json', '14d')

    def test_investigation_receipt_cannot_be_reused_for_another_scope(self):
        picked = s.choose_pairs(REFERENCE)
        report = {'version': s.VERSION, 'config': s.plan('7d'), 'referenceProductSha256': original.PRODUCT_SHA256,
                  'selectedGroups': {k: picked[k] for k in ('covered', 'uncovered')}, 'technicalStatus': 'passed',
                  'safeToCollectIndexedSettlements': True, 'checks': [{'control': {'passed': True}}], 'errors': [], 'unindexedOnChainEvents': []}
        with tempfile.TemporaryDirectory() as folder:
            path = Path(folder)/'investigation/result.json'
            original.atomic_json(path, report)
            self.assertEqual(s.require_investigation(folder, ROOT/'collector/settlement-reference.json.gz'), report)
            report['config']['cutoff'] += 1
            original.atomic_json(path, report)
            with self.assertRaisesRegex(ValueError, 'needs review'):
                s.require_investigation(folder, ROOT/'collector/settlement-reference.json.gz')

    def test_parallel_collector_is_rejected_and_interrupt_releases_lock(self):
        with tempfile.TemporaryDirectory() as folder:
            with self.assertRaises(KeyboardInterrupt):
                with s.single_collector(folder, '14d'):
                    with self.assertRaisesRegex(RuntimeError, 'Another staged run'):
                        with s.single_collector(folder, '30d'):
                            pass
                    raise KeyboardInterrupt()
            self.assertFalse((Path(folder)/'.staged-collector.lock').exists())

    def test_progress_reader_reads_existing_checkpoint_without_mutation(self):
        with tempfile.TemporaryDirectory() as folder:
            directory = s.run_dir(folder, '30d')
            database = original.connect(directory/'claims.sqlite3', s.plan('30d'))
            database.execute("INSERT INTO services VALUES('arb-one',102999,'collecting')")
            database.commit()
            database.close()
            before = hashlib.sha256((directory/'claims.sqlite3').read_bytes()).hexdigest()
            snapshot = s.progress_snapshot(folder, '30d')
            self.assertEqual(snapshot['downloadedClaims'], 0)
            self.assertEqual(snapshot['services'][0]['expectedClaims'], 102999)
            self.assertEqual(hashlib.sha256((directory/'claims.sqlite3').read_bytes()).hexdigest(), before)

    def test_included_pilot_reuse_has_no_repeat_download(self):
        with tempfile.TemporaryDirectory() as folder, patch.object(original, 'run', side_effect=AssertionError('Do not recollect pilot')), \
                patch.object(s, 'investigate', return_value={'technicalStatus': 'passed'}):
            result = s.execute('7d', folder, ROOT/'collector/settlement-reference.json.gz', PILOT_PATH, progress=lambda _: None)
            self.assertEqual(result['receipt']['coverage']['indexedClaims'], 26888)

    def test_reconciled_collection_does_not_unlock_network_or_predictions(self):
        result = s.readiness(s.read_json(PILOT_PATH))
        self.assertTrue(result['collectionReconciled'])
        self.assertFalse(result['fullNetworkEarningsReady'])
        self.assertFalse(result['historicalPredictionAutomaticallyEnabled'])
        self.assertEqual(result['originalComparableSampledSessions'], 1652)


if __name__ == '__main__':
    unittest.main()

```